# Análise Conjunta — Comparação dos Planos Nacionais de Inteligência Artificial

Este notebook integra uma análise acadêmica de rigor, desenvolvida no âmbito de uma pesquisa em Relações Internacionais dedicada ao estudo comparado das estratégias nacionais de inteligência artificial. Diferentemente dos notebooks das pastas de cada país/bloco — cada um deles dedicado à análise individual de um único documento —, este notebook tem por objeto o **conjunto de todos os principais planos e estratégias nacionais de inteligência artificial** reunidos neste projeto (Brasil, China, Estados Unidos, Europa e Índia), a fim de compará-los entre si.

O objetivo desta análise conjunta é compreender, a partir da comparação entre os documentos, quais são as **similaridades** e as **especificidades** de cada plano nacional, quais **estratégias e vertentes** — regulatórias, econômicas, de segurança nacional, de inovação tecnológica, de inclusão social, entre outras — são adotadas de forma majoritária por cada país/bloco, e em que medida essas escolhas convergem ou divergem entre si no cenário internacional.

Para tanto, a análise articula, de modo simultaneamente **quantitativo e qualitativo**, a linguagem empregada por cada documento, de forma a mapear os campos semânticos, os termos e as ênfases retóricas que cada país/bloco privilegia, e a partir dessa leitura posicionar cada plano nacional dentro do debate internacional sobre desenvolvimento e governança da inteligência artificial.

As análises e visualizações produzidas neste notebook seguem as diretrizes metodológicas da **Skill02** (Análise e Visualização Gráfica de Documentos): utilizam, de forma conjunta, os JSONs de extração de todos os países/blocos (Brasil, China, Estados Unidos, Europa e Índia), produzidos na etapa anterior (Skill01), sempre a partir do campo `texto_completo` de cada um. Em razão da diferença de extensão entre os documentos analisados, as comparações entre eles são conduzidas de forma **relativa e normalizada** (proporcional ao tamanho de cada documento), e não por meio de valores absolutos — único contexto do projeto em que esse tipo de comparação relativa entre documentos distintos é realizado. Cada visualização construída é acompanhada de sua respectiva descrição, leitura, interpretação e eventuais limitações metodológicas, com rigor acadêmico integral e fidelidade ao conteúdo original de cada documento.


## Metodologia desta execução (Skill 02_Análise_Vocab_B, revisão 2026-08-05)

Esta execução implementa integralmente a **Skill 02_Análise_Vocab_B** revisada, que corrige uma imprecisão
metodológica identificada nesta mesma revisão: o corte de **Top 50** termos por documento (herdado da
Skill 02_Análise_Vocab_A) é válido **apenas para visualizações de frequência por documento**, mas **nunca**
como base do cálculo de similaridade entre documentos. Truncar por rank antes de calcular similaridade
zera termos de forma assimétrica entre documentos, favorece o vocabulário genérico compartilhado por todos
os planos (inflando artificialmente a similaridade) e tem cobertura desigual conforme a riqueza lexical de
cada documento (demonstrado empiricamente na Seção "Riqueza lexical", abaixo).

Por isso, este notebook mantém duas bases vocabulares **estritamente separadas**, conforme o item 5 da
Skill B:

- **(a) Vocabulário completo pós-etapa 3** (sem corte de rank) — usado exclusivamente para a matriz de
  similaridade (item 6) e para toda a sequência estruturada de comparação (item 6.1).
- **(b) Corte de Top 50** — usado apenas nas visualizações de frequência por documento (pequenos múltiplos,
  ao final deste notebook), nunca como entrada da similaridade.

O pipeline de extração/remoção/normalização/desambiguação (etapas 1-3, alíneas a-b) de cada um dos seis
documentos é **reaproveitado fielmente** dos notebooks individuais já auditados sob a Skill 02_Análise_Vocab_A
(`ai_plus.ipynb`, `new_generation.ipynb`, `winning_race.ipynb`, `ai_apply_strategy.ipynb`,
`ai_continent_action_plan.ipynb`, `pbia.ipynb`) — nenhuma decisão de remoção, normalização morfológica,
tratamento de hífen/bigrama ou desambiguação contextual já tomada nesses notebooks é alterada aqui.
Duas camadas adicionais, exclusivas desta análise conjunta, são aplicadas por cima desse resultado:

1. **Exclusão de identificadores idiossincráticos de nacionalidade/instituição** (etapa 3-c da Skill B);
2. **Harmonização de rótulos entre documentos** — uma extensão da auditoria de consistência já iniciada
   nos registros individuais (Seção 9 de cada `registro_vocabulario_*.md`), que identificou uma inconsistência
   crítica não corrigida por essa auditoria anterior (ver Seção "Achado crítico", abaixo).

### Idioma dos documentos (etapa 1)

Os seis documentos comparados estão, em `texto_completo`, integralmente em **inglês** — inclusive o PBIA
(publicação oficial do MCTI/CGEE já publicada nessa língua) e os dois documentos chineses (traduções
oficiais/institucionais). Como todos os seis documentos estão no mesmo idioma, a comparação lexical direta
(após remoção e normalização) é válida, **sem necessidade de etapa de equivalência/tradução entre idiomas**
(etapa 1 da Skill B).

In [ ]:
import json, re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 120)

## Etapas 1-3: carregamento e pipeline de vocabulário por documento

As seis funções abaixo reproduzem, termo a termo, o pipeline de pré-processamento já implementado e
auditado em cada notebook individual (protocolo de uso do JSON da Skill02: apenas `titulo`, `pais_ou_bloco`
e `texto_completo`). A única alteração em relação aos notebooks de origem é que aqui **não se aplica o
corte de Top 50** — o `Counter` retornado é o vocabulário **completo** de cada documento (etapa 5-a da
Skill B).

In [ ]:
import json, re
from collections import Counter

BASE = ".."  # este notebook está em .../Análise Documentos IA/Análise Conjunta/

# ---------------------------------------------------------------------------
# Cada função reproduz FIELMENTE o pipeline já implementado e auditado no
# notebook individual do respectivo documento (Skill 02_Análise_Vocab_A já
# executada). Nenhuma decisão de remoção/normalização/bigrama/desambiguação
# já tomada nesses notebooks é alterada aqui — apenas removemos o corte de
# Top 50 (etapa 5-b da Skill B) para obter o vocabulário COMPLETO (etapa 5-a).
# ---------------------------------------------------------------------------

def load_text(relpath):
    with open(f"{BASE}/{relpath}", encoding="utf-8") as f:
        d = json.load(f)
    return d["titulo"], d["pais_ou_bloco"], d["texto_completo"]


# ============================== AI_PLUS (China) =============================
def freq_ai_plus():
    _, _, texto = load_text("China/ai_plus.json")
    STOPWORDS_EN = set("""
    a an the
    and or but nor so yet if because while although that which who whom whose when where how than whether as
    this these those it its they their them theirs he she his her him himself herself itself themselves
    we our ours us you your yours i my mine
    is are was were be been being am
    has have had having
    do does did doing
    will would shall should can could may might must
    to of in on at by for with from into through across throughout under over without within among between via per about upon toward towards up out
    not no nor
    also more most many much very only further therefore thus however moreover
    there here
    such other others any all both each every some
    including include includes included
    etc eg ie
    """.split())
    STRUCTURAL_RESIDUES = {"ii", "iii", "iv"}

    def preprocess(text):
        text = re.sub(r'Artificial [Ii]ntelligence\+', ' AIPLUSTOKEN ', text)
        text = text.replace('AI+', ' AIPLUSTOKEN ')
        text = re.sub(r'\bState Council\b', ' STATECOUNCILTOKEN ', text)
        text = re.sub(r'\b(?:factors?|relations?|paradigms?)\s+of\s+production\b', ' PRODUCTIONTHEORYTOKEN ', text, flags=re.I)
        text = re.sub(r'\bResearch and Development\b|\bR&D\b', ' RESEARCHDEVTOKEN ', text, flags=re.I)
        return text

    texto_tratado = preprocess(texto)
    tokens_brutos = re.findall(r"[A-Za-z]+(?:-[A-Za-z]+)*", texto_tratado)
    tokens_lower = [t.lower() for t in tokens_brutos]
    tokens_filtrados = [t for t in tokens_lower if len(t) > 1 and t not in STOPWORDS_EN and t not in STRUCTURAL_RESIDUES]

    NOUN_MERGES = {
        "application": ["application", "applications"], "models": ["model", "models"],
        "services": ["service", "services"], "systems": ["system", "systems"],
        "capabilities": ["capability", "capabilities"], "industry": ["industry", "industries"],
        "products": ["product", "products"], "resources": ["resource", "resources"],
        "technology": ["technology", "technologies"], "talent": ["talent", "talents"],
        "risks": ["risk", "risks"], "levels": ["level", "levels"], "agents": ["agent", "agents"],
        "initiatives": ["initiative", "initiatives"], "sectors": ["sector", "sectors"],
        "structures": ["structure", "structures"], "processes": ["process", "processes"],
        "policies": ["policy", "policies"], "ecosystems": ["ecosystem", "ecosystems"],
        "evaluations": ["evaluation", "evaluations"], "platforms": ["platform", "platforms"],
        "approach": ["approach", "approaches"],
        "governments": ["government", "governments"], "commissions": ["commission", "commissions"],
        "forces": ["force", "forces"], "transformations": ["transformation", "transformations"],
        "advantages": ["advantage", "advantages"], "scenarios": ["scenario", "scenarios"],
        "achievements": ["achievement", "achievements"], "breakthroughs": ["breakthrough", "breakthroughs"],
        "sciences": ["science", "sciences"], "architectures": ["architecture", "architectures"],
        "chains": ["chain", "chains"], "operations": ["operation", "operations"], "jobs": ["job", "jobs"],
        "environments": ["environment", "environments"], "students": ["student", "students"],
        "roles": ["role", "roles"], "humans": ["human", "humans"], "rights": ["right", "rights"],
        "incentives": ["incentive", "incentives"], "networks": ["network", "networks"],
        "investments": ["investment", "investments"], "laws": ["law", "laws"],
    }
    VERB_MERGES = {
        "promote": ["promote", "promotes", "promoting", "promoted"],
        "strengthen": ["strengthen", "strengthens", "strengthening", "strengthened"],
        "accelerate": ["accelerate", "accelerates", "accelerating", "accelerated"],
        "support": ["support", "supports", "supporting", "supported"],
        "build": ["build", "builds", "building", "built"],
        "develop": ["develop", "develops", "developing", "developed"],
        "create": ["create", "creates", "creating", "created"],
        "establish": ["establish", "establishes", "establishing", "established"],
        "improve": ["improve", "improves", "improving", "improved"],
        "enhance": ["enhance", "enhances", "enhancing", "enhanced"],
        "explore": ["explore", "explores", "exploring", "explored"],
        "encourage": ["encourage", "encourages", "encouraging", "encouraged"],
        "deepen": ["deepen", "deepens", "deepening", "deepened"],
        "optimize": ["optimize", "optimizes", "optimizing", "optimized"],
        "implement": ["implement", "implements", "implementing", "implemented"],
        "learn": ["learn", "learns", "learning", "learned"],
        "empower": ["empower", "empowers", "empowering", "empowered"],
        "exploit": ["exploit", "exploits", "exploiting", "exploited"],
        "focus": ["focus", "focuses", "focusing", "focused"],
        "foster": ["foster", "fosters", "fostering", "fostered"],
        "fund": ["fund", "funds", "funding", "funded"],
        "grow": ["grow", "grows", "growing"],
        "help": ["help", "helps", "helping", "helped"],
        "launch": ["launch", "launches", "launching", "launched"],
        "work": ["work", "works", "working", "worked"],
    }
    canon = {}
    for label, variants in NOUN_MERGES.items():
        for v in variants: canon[v] = label
    for label, variants in VERB_MERGES.items():
        for v in variants: canon[v] = label
    canon["aiplustoken"] = "AI+"
    canon["statecounciltoken"] = "State Council"
    canon["productiontheorytoken"] = "production (fatores/relações teóricas)"
    canon["ai"] = "AI"
    canon["researchdevtoken"] = "Research & Development (R&D)"

    tokens_finais = [canon.get(t, t) for t in tokens_filtrados]
    return Counter(tokens_finais)


# ======================== NEW_GENERATION (China) ============================
def freq_new_generation():
    _, _, texto = load_text("China/new_generation_ai_development_plan.json")
    STOPWORDS_EN = set("""
    a an the
    and or but nor so yet if because while although that which who whom whose when where how than whether as
    this these those it its they their them theirs he she his her him himself herself itself themselves
    we our ours us you your yours i my mine
    is are was were be been being am
    has have had having
    do does did doing
    will would shall should can could may might must
    to of in on at by for with from into through across throughout under over without within among between via per about upon toward towards
    not no nor
    also more most many much very only further therefore thus however moreover
    there here
    such other others any all both each every some
    including include includes included
    etc eg ie
    """.split())

    def preprocess(text):
        text = re.sub(r'\bartificial intelligence\b', 'AI', text, flags=re.I)
        text = re.sub(r'\bState Council\b', ' STATECOUNCILTOKEN ', text)
        text = re.sub(r'\bdriving\s+force\b', ' DRIVEFIGTOKEN ', text, flags=re.I)
        text = re.sub(r'\bdriving\s+effect\b', ' DRIVEFIGTOKEN ', text, flags=re.I)
        text = re.sub(r'\bdriving\s+role\b', ' DRIVEFIGTOKEN ', text, flags=re.I)
        text = re.sub(r'\bdriving\s+networks\b', ' DRIVEFIGTOKEN ', text, flags=re.I)
        text = re.sub(r'\bdrive\b', ' DRIVEFIGTOKEN ', text, flags=re.I)
        text = re.sub(r'(?<!-)\bdriven\b(?!-)', ' DRIVEFIGTOKEN ', text, flags=re.I)
        text = re.sub(r'\bnuclear\s+power\b', ' POWERENERGYTOKEN ', text, flags=re.I)
        text = re.sub(r'\bco-ordination\b', 'coordination', text, flags=re.I)
        text = re.sub(r'(?<![A-Za-z-])rain-inspired\b', 'brain-inspired', text, flags=re.I)
        text = re.sub(r'\bResearch and Development\b|\bR&D\b', ' RESEARCHDEVTOKEN ', text, flags=re.I)
        text = re.sub(r'\bData Infrastructure\b', ' DATAINFRATOKEN ', text, flags=re.I)
        text = re.sub(r'\bPublic Services?\b', ' PUBSERVICETOKEN ', text, flags=re.I)
        text = re.sub(r'\bValue Chains?\b', ' VALUECHAINTOKEN ', text, flags=re.I)
        text = re.sub(r'\bMachine Learning\b', ' MACHLEARNTOKEN ', text, flags=re.I)
        return text

    texto_tratado = preprocess(texto)
    tokens_brutos = re.findall(r"[A-Za-z]+(?:-[A-Za-z]+)*", texto_tratado)
    tokens_lower = [t.lower() for t in tokens_brutos]
    tokens_filtrados = [t for t in tokens_lower if len(t) > 1 and t not in STOPWORDS_EN]

    NOUN_MERGES = {
        "technology": ["technology", "technologies"], "systems": ["system", "systems"],
        "applications": ["application", "applications"], "theory": ["theory", "theories"],
        "platforms": ["platform", "platforms"], "enterprises": ["enterprise", "enterprises"],
        "breakthroughs": ["breakthrough", "breakthroughs"], "products": ["product", "products"],
        "robots": ["robot", "robots"], "capabilities": ["capability", "capabilities"],
        "bases": ["base", "bases"], "resources": ["resource", "resources"],
        "mechanisms": ["mechanism", "mechanisms"], "tasks": ["task", "tasks"],
        "projects": ["project", "projects"], "demonstrations": ["demonstration", "demonstrations"],
        "areas": ["area", "areas"], "methods": ["method", "methods"], "models": ["model", "models"],
        "standards": ["standard", "standards"], "fields": ["field", "fields"],
        "policies": ["policy", "policies"], "regulations": ["regulation", "regulations"],
        "laws": ["law", "laws"], "talent": ["talent", "talents"], "levels": ["level", "levels"],
        "results": ["result", "results"], "sectors": ["sector", "sectors"],
        "environments": ["environment", "environments"], "services": ["service", "services"],
        "networks": ["network", "networks"], "demands": ["demand", "demands"],
        "disciplines": ["discipline", "disciplines"], "domains": ["domain", "domains"],
        "plans": ["plan", "plans"], "changes": ["change", "changes"],
        "structures": ["structure", "structures"], "risks": ["risk", "risks"],
        "clusters": ["cluster", "clusters"], "forces": ["force", "forces"],
        "advantages": ["advantage", "advantages"], "reforms": ["reform", "reforms"],
        "livelihoods": ["livelihood", "livelihoods"], "chains": ["chain", "chains"],
        "teams": ["team", "teams"], "centers": ["center", "centers"],
        "frameworks": ["framework", "frameworks"], "engines": ["engine", "engines"],
        "sensors": ["sensor", "sensors"], "counterparts": ["counterpart", "counterparts"],
        "architectures": ["architecture", "architectures"], "vehicles": ["vehicle", "vehicles"],
        "markets": ["market", "markets"], "terminals": ["terminal", "terminals"],
        "trials": ["trial", "trials"], "solutions": ["solution", "solutions"],
        "organizations": ["organization", "organizations"], "assistants": ["assistant", "assistants"],
        "programs": ["program", "programs"], "incentives": ["incentive", "incentives"],
    }
    VERB_MERGES = {
        "strengthen": ["strengthen", "strengthens", "strengthening", "strengthened"],
        "promote": ["promote", "promotes", "promoting", "promoted"],
        "establish": ["establish", "establishes", "establishing", "established"],
        "develop": ["develop", "develops", "developing", "developed"],
        "construct": ["construct", "constructs", "constructing", "constructed"],
        "build": ["build", "builds", "built"],
        "accelerate": ["accelerate", "accelerates", "accelerating", "accelerated"],
        "encourage": ["encourage", "encourages", "encouraging", "encouraged"],
        "improve": ["improve", "improves", "improving", "improved"],
        "achieve": ["achieve", "achieves", "achieving", "achieved"],
        "enhance": ["enhance", "enhances", "enhancing"],
        "form": ["form", "forms", "forming", "formed"],
        "launch": ["launch", "launches", "launching", "launched"],
        "focus": ["focus", "focuses", "focusing", "focused"],
        "integrate": ["integrate", "integrates", "integrating", "integrated"],
        "coordinate": ["coordinate", "coordinates", "coordinating", "coordinated"],
        "lead": ["lead", "leads", "leading", "led"],
        "create": ["create", "creates", "creating", "created"],
    }
    canon = {}
    for label, variants in NOUN_MERGES.items():
        for v in variants: canon[v] = label
    for label, variants in VERB_MERGES.items():
        for v in variants: canon[v] = label
    canon["ai"] = "AI"
    canon["china"] = "China"
    canon["statecounciltoken"] = "State Council"
    canon["drivefigtoken"] = "driving/drive (força motriz)"
    canon["powerenergytoken"] = "power (nuclear/energia)"
    canon["power"] = "power (nacional/geopolítico)"
    canon["driving"] = "driving (condução autônoma)"
    canon["researchdevtoken"] = "Research & Development (R&D)"
    canon["datainfratoken"] = "Data Infrastructure"
    canon["pubservicetoken"] = "Public Service(s)"
    canon["valuechaintoken"] = "Value Chain"
    canon["machlearntoken"] = "Machine Learning"

    tokens_finais = [canon.get(t, t) for t in tokens_filtrados]
    return Counter(tokens_finais)


# ==================== AMERICAS_AI_ACTION_PLAN (EUA) =========================
def freq_americas():
    _, _, texto = load_text("Estados Unidos/americas_ai_action_plan.json")
    STOPWORDS_EN = set("""
    a an the
    and or but nor so yet if because while although that which who whom whose when where how than whether as
    this these those it its they their them theirs he she his her him himself herself itself themselves
    we our ours us you your yours i my mine
    is are was were be been being am
    has have had having
    do does did doing
    will would shall should can could may might must
    to of in on at by for with from into through across throughout under over without within among between via per about upon toward towards
    not no nor
    also more most many much very only further therefore thus however moreover
    there here
    such other others any all both each every some
    including include includes included
    etc eg ie
    """.split())
    STRUCTURAL_PHRASES_REMOVIDAS = [r'\bRecommended Policy Actions\b']

    def preprocess(text):
        text = text.replace("U.S.", "United States")
        text = re.sub(r'\bArtificial Intelligence\b', 'AI', text, flags=re.I)
        for pat in STRUCTURAL_PHRASES_REMOVIDAS:
            text = re.sub(pat, ' ', text)
        text = re.sub(r'\bUnited States\b', ' UNITEDSTATESTOKEN ', text)
        text = re.sub(r'\bpower\s+generation\b', ' POWERGENTOKEN ', text, flags=re.I)
        text = re.sub(r'\benergy\s+generation\b', ' POWERGENTOKEN ', text, flags=re.I)
        text = re.sub(r'\bcomputing\s+power\b', ' POWERCOMPUTETOKEN ', text, flags=re.I)
        text = re.sub(r'\bbalance\s+of\s+power\b', ' POWERGEOTOKEN ', text, flags=re.I)
        text = re.sub(r'\bpower\s+of\s+American\s+innovation\b', ' POWERGEOTOKEN ', text, flags=re.I)
        text = re.sub(r'\bResearch and Development\b|\bR&D\b', ' RESEARCHDEVTOKEN ', text, flags=re.I)
        text = re.sub(r'\bData Centers?\b', ' DATACENTERTOKEN ', text, flags=re.I)
        text = re.sub(r'\bprivate-sector(?!-)\b', 'private sector', text, flags=re.I)
        text = re.sub(r'\bprivate sector\b', ' PRIVATESECTORTOKEN ', text, flags=re.I)
        text = re.sub(r'\bFederal Government\b', ' FEDGOVTOKEN ', text, flags=re.I)
        return text

    texto_tratado = preprocess(texto)
    tokens_brutos = re.findall(r"[A-Za-z]+(?:-[A-Za-z]+)*", texto_tratado)
    tokens_lower = [t.lower() for t in tokens_brutos]
    tokens_filtrados = [t for t in tokens_lower if len(t) > 1 and t not in STOPWORDS_EN]

    NOUN_MERGES = {
        "systems": ["system", "systems"], "models": ["model", "models"], "agencies": ["agency", "agencies"],
        "technology": ["technology", "technologies"], "programs": ["program", "programs"], "workers": ["worker", "workers"],
        "controls": ["control", "controls"], "standards": ["standard", "standards"], "capabilities": ["capability", "capabilities"],
        "tools": ["tool", "tools"], "actions": ["action", "actions"], "risks": ["risk", "risks"],
        "developers": ["developer", "developers"], "initiatives": ["initiative", "initiatives"], "stakeholders": ["stakeholder", "stakeholders"],
        "frameworks": ["framework", "frameworks"], "resources": ["resource", "resources"], "occupations": ["occupation", "occupations"],
        "assessments": ["assessment", "assessments"], "evaluations": ["evaluation", "evaluations"], "regulations": ["regulation", "regulations"],
        "values": ["value", "values"], "threats": ["threat", "threats"], "vulnerabilities": ["vulnerability", "vulnerabilities"],
        "centers": ["center", "centers"], "innovation": ["innovation", "innovations"], "needs": ["need", "needs"],
        "skills": ["skill", "skills"], "countries": ["country", "countries"], "efforts": ["effort", "efforts"],
        "employers": ["employer", "employers"], "industry": ["industry", "industries"], "partners": ["partner", "partners"],
        "sector": ["sector", "sectors"], "services": ["service", "services"], "sources": ["source", "sources"],
        "states (subnacionais, distinto de United States)": ["state", "states"],
        "fields": ["field", "fields"], "benefits": ["benefit", "benefits"], "pillars": ["pillar", "pillars"],
        "semiconductors": ["semiconductor", "semiconductors"], "applications": ["application", "applications"],
        "jobs": ["job", "jobs"], "goals": ["goal", "goals"], "laws": ["law", "laws"],
        "businesses": ["business", "businesses"], "rules": ["rule", "rules"], "orders": ["order", "orders"],
        "governments": ["government", "governments"], "academics": ["academic", "academics"],
        "markets": ["market", "markets"], "investments": ["investment", "investments"],
        "projects": ["project", "projects"], "approaches": ["approach", "approaches"], "chips": ["chip", "chips"],
        "environments": ["environment", "environments"], "processes": ["process", "processes"],
        "workflows": ["workflow", "workflows"], "deepfakes": ["deepfake", "deepfakes"], "ways": ["way", "ways"],
        "officers": ["officer", "officers"], "nations": ["nation", "nations"], "agendas": ["agenda", "agendas"],
        "allies": ["ally", "allies", "allied"], "challenges": ["challenge", "challenges"], "councils": ["council", "councils"],
        "protections": ["protection", "protections"], "requirements": ["requirement", "requirements"],
        "reviews": ["review", "reviews"], "roles": ["role", "roles"], "providers": ["provider", "providers"],
    }
    VERB_MERGES = {
        "lead / led": ["lead", "leads", "leading", "led"],
        "develop": ["develop", "develops", "developing", "developed"],
        "create": ["create", "creates", "creating", "created"],
        "build": ["build", "builds", "building", "built"],
        "establish": ["establish", "establishes", "establishing", "established"],
        "ensure": ["ensure", "ensures", "ensuring", "ensured"],
        "promote": ["promote", "promotes", "promoting", "promoted"],
        "expand": ["expand", "expands", "expanding", "expanded"],
        "support": ["support", "supports", "supporting", "supported"],
        "make": ["make", "makes", "making", "made"],
        "require": ["require", "requires", "requiring", "required"],
        "use": ["use", "uses", "used", "using"],
        "adopt": ["adopt", "adopts", "adopting", "adopted"],
        "align": ["align", "aligns", "aligning", "aligned"],
        "conduct": ["conduct", "conducts", "conducting", "conducted"],
        "deliver": ["deliver", "delivers", "delivering", "delivered"],
        "design": ["design", "designs", "designing", "designed"],
        "direct": ["direct", "directs", "directing", "directed"],
        "fund": ["fund", "funds", "funding", "funded"],
        "implement": ["implement", "implements", "implementing", "implemented"],
        "increase": ["increase", "increases", "increasing", "increased"],
        "launch": ["launch", "launches", "launching", "launched"],
        "maintain": ["maintain", "maintains", "maintaining", "maintained"],
        "measure": ["measure", "measures", "measuring", "measured"],
        "pilot": ["pilot", "pilots", "piloting", "piloted"],
        "protect": ["protect", "protects", "protecting", "protected"],
        "streamline": ["streamline", "streamlines", "streamlining", "streamlined"],
        "supply": ["supply", "supplies", "supplying", "supplied"],
        "train": ["train", "trains", "training", "trained"],
        "transform": ["transform", "transforms", "transforming", "transformed"],
    }
    canon = {}
    for label, variants in NOUN_MERGES.items():
        for v in variants: canon[v] = label
    for label, variants in VERB_MERGES.items():
        for v in variants: canon[v] = label
    canon["ai"] = "AI"
    canon["unitedstatestoken"] = "United States"
    canon["doc"] = "DOC"
    canon["powergentoken"] = "power/energy generation (infraestrutura elétrica)"
    canon["powercomputetoken"] = "power (computacional)"
    canon["powergeotoken"] = "power (geopolítico/abstrato)"
    canon["power"] = "power (energia/infraestrutura elétrica)"
    canon["generation"] = "generation (tecnológica)"
    canon["researchdevtoken"] = "Research & Development (R&D)"
    canon["datacentertoken"] = "Data Center(s)"
    canon["privatesectortoken"] = "Private Sector"
    canon["fedgovtoken"] = "Federal Government"

    tokens_finais = [canon.get(t, t) for t in tokens_filtrados]
    return Counter(tokens_finais)


# ========================= APPLY_AI_STRATEGY (UE) ============================
def freq_apply_ai():
    _, _, texto = load_text("Europa/apply_ai_strategy.json")
    STOPWORDS_EN = set("""
    a an the
    and or but nor so yet if because while although that which who whom whose when where how than whether as
    this these those it its they their them theirs he she his her him himself herself itself themselves
    we our ours us you your yours i my mine
    is are was were be been being am
    has have had having
    do does did doing
    will would shall should can could may might must
    to of in on at by for with from into through across throughout under over without within among between via per about upon toward towards
    not no nor
    also more most many much very only further therefore thus however moreover
    there here
    such other others any all both each every some
    including include includes included
    etc eg ie e g
    s
    """.split())

    def preprocess(text):
        text = re.sub(r'\bArtificial Intelligence\b', 'AI', text, flags=re.I)
        text = re.sub(r'\bEuropean Union\b', 'EU', text, flags=re.I)
        text = re.sub(r'\bMember States\b', ' MEMBERSTATESTOKEN ', text, flags=re.I)
        text = re.sub(r'\bdigital twins?\b', ' DIGITALTWINSTOKEN ', text, flags=re.I)
        text = re.sub(r'\bAI Act\b', ' AIACTTOKEN ', text, flags=re.I)
        text = text.replace('a building or even of a human body', 'a buildingnoun or even of a human body')
        text = re.sub(r'\bcomputing\s+power\b', ' POWERCOMPUTETOKEN ', text, flags=re.I)
        text = re.sub(r'\bpredictive\s+power\b', ' POWERABSTOKEN ', text, flags=re.I)
        text = re.sub(r'\bAction\b', ' ACTIONPROPERTOKEN ', text)
        text = re.sub(r'\bAlliance\b', ' ALLIANCEPROPERTOKEN ', text)
        text = re.sub(r'\bpublic sector\b', ' PUBLICSECTORTOKEN ', text, flags=re.I)
        return text

    NOUN_MERGES = {
        "sectors": ["sector", "sectors"], "models": ["model", "models"], "solutions": ["solution", "solutions"],
        "systems": ["system", "systems"], "tools": ["tool", "tools"], "services": ["service", "services"],
        "needs": ["need", "needs", "needed"], "actions": ["action", "actions"], "initiatives": ["initiative", "initiatives"],
        "challenges": ["challenge", "challenges"], "applications": ["application", "applications"],
        "infrastructures": ["infrastructure", "infrastructures"], "skills": ["skill", "skills"],
        "frameworks": ["framework", "frameworks"], "networks": ["network", "networks"], "platforms": ["platform", "platforms"],
        "organisations": ["organisation", "organisations"], "administrations": ["administration", "administrations"],
        "hubs": ["hub", "hubs"], "practices": ["practice", "practices"], "tasks": ["task", "tasks"],
        "robots": ["robot", "robots"], "vehicles": ["vehicle", "vehicles"], "labs": ["lab", "labs"],
        "benefits": ["benefit", "benefits"], "assets": ["asset", "assets"], "architectures": ["architecture", "architectures"],
        "flagships": ["flagship", "flagships"], "boards": ["board", "boards"], "resources": ["resource", "resources"],
        "threats": ["threat", "threats"], "dialogues": ["dialogue", "dialogues"], "developments": ["development", "developments"],
        "ecosystems": ["ecosystem", "ecosystems"], "capacities": ["capacity", "capacities"],
        "technologies": ["technology", "technologies"], "sciences": ["science", "sciences"],
        "processes": ["process", "processes"], "decisions": ["decision", "decisions"], "areas": ["area", "areas"],
        "ways": ["way", "ways"], "businesses": ["business", "businesses"], "humans": ["human", "humans"],
        "alliances": ["alliance", "alliances"], "purposes": ["purpose", "purposes"], "steps": ["step", "steps"],
        "funds": ["fund", "funds"], "risks": ["risk", "risks"], "interests": ["interest", "interests"],
        "chains": ["chain", "chains"], "startups": ["startup", "startups"], "strengths": ["strength", "strengths"],
        "professionals": ["professional", "professionals"], "drivers": ["driver", "drivers"], "factors": ["factor", "factors"],
        "partners": ["partner", "partners"], "partnerships": ["partnership", "partnerships"],
        "governments": ["government", "governments"], "forms": ["form", "forms"], "results": ["result", "results"],
        "elements": ["element", "elements"], "gaps": ["gap", "gaps"], "values": ["value", "values"],
        "innovations": ["innovation", "innovations"], "strategies": ["strategy", "strategies"],
        "industries": ["industry", "industries"],
        "buildings": ["buildingnoun", "buildings"],
        "states (genérico, não EU)": ["state", "states"],
        "sources": ["source", "sources"], "programmes": ["programme", "programmes"],
        "impacts": ["impact", "impacts"], "operations": ["operation", "operations"],
        "changes": ["change", "changes"], "communications": ["communication", "communications"],
        "advances": ["advance", "advances"], "trainings": ["training", "trainings"], "users": ["user", "users"],
    }
    VERB_MERGES = {
        "support": ["support", "supports", "supporting", "supported"],
        "foster": ["foster", "fosters", "fostering", "fostered"],
        "promote": ["promote", "promotes", "promoting", "promoted"],
        "develop (verbo)": ["develop", "develops", "developing", "developed"],
        "build": ["build", "builds", "building", "built"],
        "ensure": ["ensure", "ensures", "ensuring", "ensured"],
        "enable": ["enable", "enables", "enabling", "enabled"],
        "accelerate": ["accelerate", "accelerates", "accelerating", "accelerated"],
        "deploy": ["deploy", "deploys", "deploying", "deployed"],
        "create": ["create", "creates", "creating", "created"],
        "facilitate": ["facilitate", "facilitates", "facilitating", "facilitated"],
        "provide": ["provide", "provides", "providing", "provided"],
        "address": ["address", "addresses", "addressing", "addressed"],
        "improve": ["improve", "improves", "improving", "improved"],
        "launch": ["launch", "launches", "launching", "launched"],
        "establish": ["establish", "establishes", "establishing", "established"],
        "monitor": ["monitor", "monitors", "monitoring", "monitored"],
        "help": ["help", "helps", "helping", "helped"],
        "use": ["use", "uses", "using", "used"],
        "increase": ["increase", "increases", "increasing", "increased"],
        "boost": ["boost", "boosts", "boosting", "boosted"],
        "drive": ["drive", "drives", "driving"],
        "deliver": ["deliver", "delivers", "delivering", "delivered"],
        "leverage": ["leverage", "leverages", "leveraging", "leveraged"],
        "secure (verbo/adj)": ["secure", "secures", "securing", "secured"],
        "strengthen": ["strengthen", "strengthens", "strengthening", "strengthened"],
        "expand": ["expand", "expands", "expanding", "expanded"],
        "integrate": ["integrate", "integrates", "integrating", "integrated"],
        "encourage": ["encourage", "encourages", "encouraging", "encouraged"],
        "protect": ["protect", "protects", "protecting", "protected"],
        "reduce": ["reduce", "reduces", "reducing", "reduced"],
        "adopt": ["adopt", "adopts", "adopting", "adopted"],
        "implement": ["implement", "implements", "implementing", "implemented"],
        "reinforce": ["reinforce", "reinforces", "reinforcing", "reinforced"],
        "remain": ["remain", "remains", "remaining", "remained"],
        "focus (verbo)": ["focus", "focuses", "focusing", "focused"],
    }

    texto_tratado = preprocess(texto)
    tokens_brutos = re.findall(r"[A-Za-z_]+(?:-[A-Za-z]+)*", texto_tratado)
    tokens_lower = [t.lower() for t in tokens_brutos]
    tokens_filtrados = [t for t in tokens_lower if len(t) > 1 and t not in STOPWORDS_EN]

    canon = {}
    for label, variants in NOUN_MERGES.items():
        for v in variants: canon[v] = label
    for label, variants in VERB_MERGES.items():
        for v in variants: canon[v] = label
    canon["ai"] = "AI"
    canon["aiacttoken"] = "AI Act"
    canon["eu"] = "EU"
    canon["memberstatestoken"] = "Member States"
    canon["digitaltwinstoken"] = "digital twins"
    canon["powercomputetoken"] = "power (computacional)"
    canon["powerabstoken"] = "power (capacidade preditiva/abstrata)"
    canon["power"] = "power (energia)"
    canon["actionpropertoken"] = "Action (nome de plano/programa)"
    canon["alliancepropertoken"] = "Alliance (nome próprio de entidade)"
    canon["publicsectortoken"] = "public sector"

    tokens_finais = [canon.get(t, t) for t in tokens_filtrados]
    return Counter(tokens_finais)


# ===================== AI_CONTINENT_ACTION_PLAN (UE) =========================
def freq_ai_continent():
    _, _, texto = load_text("Europa/ai_continent_action_plan.json")
    STOPWORDS_EN = set("""
    a an the
    and or but nor so yet if because while although that which who whom whose when where how than whether as
    this these those it its they their them theirs he she his her him himself herself itself themselves
    we our ours us you your yours i my mine
    is are was were be been being am
    has have had having
    do does did doing
    will would shall should can could may might must
    to of in on at by for with from into through across throughout under over without within among between via per about upon toward towards
    not no nor
    also more most many much very only further therefore thus however moreover
    there here
    such other others any all both each every some
    including include includes included
    etc eg ie
    up well
    """.split())
    STRUCTURAL_PHRASES_REMOVIDAS = [r'\bKey Commission(?:\s*/\s*EuroHPC)?\s+[Aa]ctions:\s*']
    COMPOUND_PATTERNS = [
        (r"\bEuropean Union\b", " EU "),
        (r"\bCloud and AI Development Act\b", " CLOUD_AI_DEVELOPMENT_ACT "),
        (r"\bEuroHPC Joint Undertaking\b", " EUROHPC_JOINT_UNDERTAKING "),
        (r"\bAI Skills Academy\b", " AI_SKILLS_ACADEMY "),
        (r"\bApply AI Strategy\b", " APPLY_AI_STRATEGY "),
        (r"\bData Union Strategy\b", " DATA_UNION_STRATEGY "),
        (r"\bEuropean Digital Innovation Hubs?\b", " DIGITAL_INNOVATION_HUBS "),
        (r"\bDigital Innovation Hubs?\b", " DIGITAL_INNOVATION_HUBS "),
        (r"\bAI in Science\b", " AI_IN_SCIENCE "),
        (r"\bAI Gigafactor(?:y|ies)\b", " AI_GIGAFACTORIES "),
        (r"\bAI Factor(?:y|ies)\b", " AI_FACTORIES "),
        (r"\bAI Act\b", " AI_ACT "),
        (r"\bAI Office\b", " AI_OFFICE "),
        (r"\bMember States\b", " MEMBER_STATES "),
        (r"\bSingle Market\b", " SINGLE_MARKET "),
        (r"\bData Labs?\b", " DATA_LABS "),
        (r"\bpublic sector\b", " PUBLIC_SECTOR "),
        (r"\bprivate sector\b", " PRIVATE_SECTOR "),
        (r"\bPublic Services?\b", " PUBLIC_SERVICE "),
        (r"\bValue Chains?\b", " VALUE_CHAIN "),
        (r"\bPublic Administration\b", " PUBLIC_ADMINISTRATION "),
    ]

    def preprocess(text):
        for pat in STRUCTURAL_PHRASES_REMOVIDAS:
            text = re.sub(pat, ' ', text)
        text = re.sub(r'\bpublic-sector\b', 'public sector', text, flags=re.I)
        text = re.sub(r'\bprivate-sector\b', 'private sector', text, flags=re.I)
        for pat, repl in COMPOUND_PATTERNS:
            text = re.sub(pat, repl, text, flags=re.I)
        text = re.sub(r'\bcomput(?:ing|ational)\s+power\b', ' POWERCOMPUTETOKEN ', text, flags=re.I)
        text = re.sub(r'\bpower\s+capacity\b', ' POWERENERGYTOKEN ', text, flags=re.I)
        text = re.sub(r'\bpower\s+their\b', ' POWERENERGYTOKEN their', text, flags=re.I)
        text = re.sub(r'\bpurchasing\s+power\b', ' POWERECONTOKEN ', text, flags=re.I)
        return text

    texto_tratado = preprocess(texto)
    PLACEHOLDERS = {repl.strip() for _, repl in COMPOUND_PATTERNS} | {"AI"}
    tokens_brutos = re.findall(r"[A-Za-z_]+(?:-[A-Za-z]+)*", texto_tratado)
    tokens_lower = [t if t in PLACEHOLDERS else t.lower() for t in tokens_brutos]
    tokens_filtrados = [t for t in tokens_lower if (t in PLACEHOLDERS) or (len(t) > 1 and t not in STOPWORDS_EN)]

    NOUN_MERGES = {
        "models": ["model", "models"], "services": ["service", "services"], "solutions": ["solution", "solutions"],
        "sectors": ["sector", "sectors"], "initiatives": ["initiative", "initiatives"], "resources": ["resource", "resources"],
        "stakeholders": ["stakeholder", "stakeholders"], "facilities": ["facility", "facilities"],
        "partnerships": ["partnership", "partnerships"], "programmes": ["programme", "programmes"],
        "skills": ["skill", "skills"], "actions": ["action", "actions"], "needs": ["need", "needs"],
        "companies": ["company", "companies"], "startups": ["startup", "startups"], "scaleups": ["scaleup", "scaleups"],
        "technologies": ["technology", "technologies"], "values": ["value", "values"], "tools": ["tool", "tools"],
        "networks": ["network", "networks"], "supercomputers": ["supercomputer", "supercomputers"],
        "talent": ["talent", "talents"], "projects": ["project", "projects"], "funds": ["fund", "funds"],
        "investments": ["investment", "investments"], "gaps": ["gap", "gaps"], "spaces": ["space", "spaces"],
        "risks": ["risk", "risks"], "areas": ["area", "areas"], "chips": ["chip", "chips"],
        "strategy": ["strategy", "strategies"], "development": ["development", "developments"],
        "capacity": ["capacity", "capacities"],
        "applications": ["application", "applications"], "businesses": ["business", "businesses"],
        "domains": ["domain", "domains"], "sciences": ["science", "sciences"],
        "infrastructures": ["infrastructure", "infrastructures"], "administrations": ["administration", "administrations"],
        "provisions": ["provision", "provisions"], "calls": ["call", "calls"], "markets": ["market", "markets"],
        "ways": ["way", "ways"], "communications": ["communication", "communications"],
        "dialogues": ["dialogue", "dialogues"], "examples": ["example", "examples"], "processes": ["process", "processes"],
        "partners": ["partner", "partners"], "degrees": ["degree", "degrees"], "schemes": ["scheme", "schemes"],
        "candidates": ["candidate", "candidates"], "fields": ["field", "fields"], "centres": ["centre", "centres"],
        "efforts": ["effort", "efforts"],
    }
    VERB_MERGES = {
        "support": ["support", "supports", "supporting", "supported"],
        "launch": ["launch", "launching", "launched"],
        "ensure": ["ensure", "ensures"],
        "use": ["use", "uses", "using", "used"],
        "develop": ["develop", "developing", "developed"],
        "provide": ["provide", "provides", "providing", "provided"],
        "facilitate": ["facilitate", "facilitates", "facilitating", "facilitated"],
        "train": ["train", "training", "trained"],
        "adopt": ["adopt", "adopted"],
        "build": ["build", "building"],
        "create": ["create", "creates", "creating"],
        "strengthen": ["strengthen", "strengthening", "strengthened"],
        "increase": ["increase", "increases", "increasing", "increased"],
        "foster": ["foster", "fosters", "fostering"],
        "attract": ["attract", "attracting"],
        "work": ["work", "works", "working"],
        "set": ["set", "sets"],
        "link": ["link", "links", "linking", "linked"],
        "offer": ["offer", "offers", "offering", "offered"],
        "require": ["require", "requires"],
        "address": ["address", "addresses"],
        "identify": ["identify", "identifying"],
        "continue": ["continue", "continues"],
        "aim": ["aim", "aims", "aimed"],
        "enhance": ["enhance", "enhances"],
        "promote": ["promote", "promoting"],
        "focus (verbo)": ["focus", "focuses", "focusing", "focused"],
    }

    canon = {}
    for label, variants in NOUN_MERGES.items():
        for v in variants: canon[v] = label
    for label, variants in VERB_MERGES.items():
        for v in variants: canon[v] = label
    canon["eu"] = "EU"
    canon["powercomputetoken"] = "power (computacional)"
    canon["powerenergytoken"] = "power (energia)"
    canon["powerecontoken"] = "power (econômico/abstrato)"

    DISPLAY_LABELS = {
        "AI_FACTORIES": "AI Factories", "AI_GIGAFACTORIES": "AI Gigafactories", "MEMBER_STATES": "Member States",
        "AI_ACT": "AI Act", "DIGITAL_INNOVATION_HUBS": "Digital Innovation Hubs", "DATA_UNION_STRATEGY": "Data Union Strategy",
        "DATA_LABS": "Data Labs", "APPLY_AI_STRATEGY": "Apply AI Strategy", "AI_SKILLS_ACADEMY": "AI Skills Academy",
        "CLOUD_AI_DEVELOPMENT_ACT": "Cloud and AI Development Act", "AI_IN_SCIENCE": "AI in Science",
        "AI_OFFICE": "AI Office", "SINGLE_MARKET": "Single Market", "EUROHPC_JOINT_UNDERTAKING": "EuroHPC Joint Undertaking",
        "PUBLIC_SECTOR": "public sector", "PRIVATE_SECTOR": "private sector", "EU": "EU", "AI": "AI",
        "PUBLIC_SERVICE": "Public Service(s)", "VALUE_CHAIN": "Value Chain", "PUBLIC_ADMINISTRATION": "Public Administration",
    }

    tokens_finais = [canon.get(t, t) for t in tokens_filtrados]
    freq = Counter(tokens_finais)
    # aplica rótulos de exibição também no vocabulário completo (não só no Top 50),
    # para manter os mesmos rótulos usados no notebook individual
    freq_final = Counter()
    for k, v in freq.items():
        freq_final[DISPLAY_LABELS.get(k, k)] += v
    return freq_final


# ================================ PBIA (Brasil) ==============================
def freq_pbia():
    _, _, texto_completo = load_text("Brasil/pbia.json")

    COMPOUND_TERMS = [
        (r"\bResearch and Development\b|\bR&D\b", "ZZCMPRESEARCHDEV", "Research & Development (R&D)"),
        (r"\bUnified Health System\b|\bSUS\b", "ZZCMPSUS", "Unified Health System (SUS)"),
        (r"\bNational Data Infrastructure\b|\bIND\b", "ZZCMPIND", "National Data Infrastructure (IND)"),
        (r"\bLarge Language Models?\b|\bLLMs?\b", "ZZCMPLLM", "Large Language Models (LLM)"),
        (r"\bSustainable Development Goals?\b|\bSDGs?\b", "ZZCMPSDG", "Sustainable Development Goals (SDGs)"),
        (r"\bAI Act\b", "ZZCMPAIACT", "AI Act"),
        (r"\bArtificial Intelligence\b|\bAI\b", "ZZCMPAI", "Artificial Intelligence (AI)"),
        (r"\bData Centers?\b", "ZZCMPDATACENTER", "Data Center(s)"),
        (r"\bData Infrastructure\b", "ZZCMPDATAINFRA", "Data Infrastructure"),
        (r"\bPublic Sector\b", "ZZCMPPUBSECTOR", "Public Sector"),
        (r"\bPublic Services?\b", "ZZCMPPUBSERVICE", "Public Service(s)"),
        (r"\bPrivate Sector\b", "ZZCMPPRIVSECTOR", "Private Sector"),
        (r"\bValue Chains?\b", "ZZCMPVALUECHAIN", "Value Chain"),
        (r"\bMachine Learning\b", "ZZCMPMACHLEARN", "Machine Learning"),
        (r"\bFederal Government\b", "ZZCMPFEDGOV", "Federal Government"),
        (r"\bDigital Government\b", "ZZCMPDIGGOV", "Digital Government"),
        (r"\bPublic Administration\b", "ZZCMPPUBADMIN", "Public Administration"),
        (r"\bClean Energy\b", "ZZCMPCLEANENERGY", "Clean Energy"),
        (r"\bEnergy Matrix\b", "ZZCMPENERGYMATRIX", "Energy Matrix"),
    ]
    CONTEXT_DISAMBIGUATION = [
        (r"\bUnited States\b", "ZZCMPUNITEDSTATES", "United States", False),
        (r"\bthe State\b", "ZZCMPSTATEENTITY", "the State (ente político/entidade abstrata)", False),
        (r"\bFederal Public Power\b", "ZZCMPPUBLICPOWER", "Poder Público Federal (termo jurídico-institucional)", False),
        (r"\bGeneration of national\b", "ZZCMPGENAXIS", "Generation of national capacities/capabilities (título de eixo do Plano)", False),
        (r"\bcomputational power\b", "ZZCMPPOWERCOMPUTE", "power (computacional)", True),
        (r"\bcontent generation\b", "ZZCMPGENCONTENT", "generation (geração de conteúdo por IA)", True),
    ]

    work_text = texto_completo
    placeholder_map = {}
    for pattern, placeholder, label in COMPOUND_TERMS:
        placeholder_map[placeholder.lower()] = label
        work_text = re.sub(pattern, f" {placeholder} ", work_text, flags=re.IGNORECASE)
    for pattern, placeholder, label, case_insensitive in CONTEXT_DISAMBIGUATION:
        placeholder_map[placeholder.lower()] = label
        flags = re.IGNORECASE if case_insensitive else 0
        work_text = re.sub(pattern, f" {placeholder} ", work_text, flags=flags)

    raw_tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ'\-]*", work_text)
    tokens = [t.lower() for t in raw_tokens]

    ARTICLES = {"a", "an", "the"}
    PREPOSITIONS = {"of","in","to","for","with","on","by","as","at","from","into","about","through",
        "during","before","after","above","below","between","under","over","without","within","among",
        "throughout","towards","toward","upon","across","per","via","despite","unlike","regarding","off",
        "out","up","down","besides"}
    CONJUNCTIONS = {"and","or","but","nor","so","yet","if","because","while","although","though",
        "whether","since","unless","until","than"}
    PRONOUNS_DETERMINERS = {"it","its","this","that","these","those","we","our","ours","they","their",
        "theirs","which","who","whom","whose","i","you","he","she","him","her","us","them","his","hers",
        "itself","themselves","ourselves","yourself","yourselves","himself","herself","one","ones",
        "such","other","others","any","some","each","all","both","either","neither","no","none","own",
        "same"}
    AUX_MODAL_VERBS = {"is","are","was","were","be","been","being","am","has","have","had","do","does",
        "did","will","would","shall","should","can","could","may","might","must","ought"}
    GENERIC_CONNECTORS = {"more","most","much","many","few","several","various","not","also","only",
        "just","still","even","well","thus","therefore","however","moreover","furthermore","given",
        "whereas"}
    STOPWORDS = (ARTICLES | PREPOSITIONS | CONJUNCTIONS | PRONOUNS_DETERMINERS
                 | AUX_MODAL_VERBS | GENERIC_CONNECTORS)
    LIST_MARKERS = {"b", "c", "d", "ii", "iii", "iv", "v"}
    TEMPLATE_TERMS = {"expected"}

    PLURAL_MERGE = {
        "actions": "action(s)", "action": "action(s)",
        "challenges": "challenge(s)", "challenge": "challenge(s)",
        "impacts": "impact(s)", "impact": "impact(s)",
        "solutions": "solution(s)", "solution": "solution(s)",
        "resources": "resource(s)", "resource": "resource(s)",
        "investments": "investment(s)", "investment": "investment(s)",
        "models": "model(s)", "model": "model(s)",
        "benefits": "benefit(s)", "benefit": "benefit(s)",
        "professionals": "professional(s)", "professional": "professional(s)",
        "networks": "network(s)", "network": "network(s)",
        "initiatives": "initiative(s)", "initiative": "initiative(s)",
        "institutions": "institution(s)", "institution": "institution(s)",
        "researchers": "researcher(s)", "researcher": "researcher(s)",
        "capacities": "capacity/capacities", "capacity": "capacity/capacities",
        "companies": "company/companies", "company": "company/companies",
        "citizens": "citizen(s)", "citizen": "citizen(s)",
        "agencies": "agency/agencies", "agency": "agency/agencies",
        "axes": "axis/axes", "axis": "axis/axes",
        "centers": "center(s)", "center": "center(s)",
        "innovations": "innovation(s)", "innovation": "innovation(s)",
        "qualifications": "qualification(s)", "qualification": "qualification(s)",
        "systems": "system(s)", "system": "system(s)",
        "technologies": "technology/technologies", "technology": "technology/technologies",
        "sectors": "sector(s)", "sector": "sector(s)",
        "processes": "process(es)", "process": "process(es)",
        "services": "service(s)", "service": "service(s)",
        "governments": "government(s)", "government": "government(s)",
        "policies": "policy/policies", "policy": "policy/policies",
        "rights": "right(s)", "right": "right(s)",
        "risks": "risk(s)", "risk": "risk(s)",
        "servants": "servant(s)", "servant": "servant(s)",
        "brazil's": "brazil",
        "applications": "application(s)", "application": "application(s)",
        "areas": "area(s)", "area": "area(s)",
        "advantages": "advantage(s)", "advantage": "advantage(s)",
        "frameworks": "framework(s)", "framework": "framework(s)",
        "databases": "database(s)", "database": "database(s)",
        "decisions": "decision(s)", "decision": "decision(s)",
        "ecosystems": "ecosystem(s)", "ecosystem": "ecosystem(s)",
        "enterprises": "enterprise(s)", "enterprise": "enterprise(s)",
        "environments": "environment(s)", "environment": "environment(s)",
        "examples": "example(s)", "example": "example(s)",
        "funds": "fund(s)", "fund": "fund(s)",
        "goals": "goal(s)", "goal": "goal(s)",
        "humans": "human(s)", "human": "human(s)",
        "interests": "interest(s)", "interest": "interest(s)",
        "jobs": "job(s)", "job": "job(s)",
        "leaders": "leader(s)", "leader": "leader(s)",
        "levels": "level(s)", "level": "level(s)",
        "markets": "market(s)", "market": "market(s)",
        "missions": "mission(s)", "mission": "mission(s)",
        "nations": "nation(s)", "nation": "nation(s)",
        "objectives": "objective(s)", "objective": "objective(s)",
        "partnerships": "partnership(s)", "partnership": "partnership(s)",
        "plans": "plan(s)", "plan": "plan(s)",
        "platforms": "platform(s)", "platform": "platform(s)",
        "populations": "population(s)", "population": "population(s)",
        "problems": "problem(s)", "problem": "problem(s)",
        "projects": "project(s)", "project": "project(s)",
        "scenarios": "scenario(s)", "scenario": "scenario(s)",
        "schools": "school(s)", "school": "school(s)",
        "sciences": "science(s)", "science": "science(s)",
        "standards": "standard(s)", "standard": "standard(s)",
        "students": "student(s)", "student": "student(s)",
        "supercomputers": "supercomputer(s)", "supercomputer": "supercomputer(s)",
        "tools": "tool(s)", "tool": "tool(s)",
        "transformations": "transformation(s)", "transformation": "transformation(s)",
        "treatments": "treatment(s)", "treatment": "treatment(s)",
        "volumes": "volume(s)", "volume": "volume(s)",
        "contexts": "context(s)", "context": "context(s)",
        "characteristics": "characteristic(s)", "characteristic": "characteristic(s)",
        "biases": "bias(es)", "bias": "bias(es)",
        "businesses": "business(es)", "business": "business(es)",
        "costs": "cost(s)", "cost": "cost(s)",
        "changes": "change(s)", "change": "change(s)",
        "results": "result(s)", "result": "result(s)",
        "states": "state(s) (subnacional/genérico)", "state": "state(s) (subnacional/genérico)",
    }
    VERB_MERGE = {
        "promote": "promote", "promotes": "promote", "promoting": "promote", "promoted": "promote",
        "increase": "increase", "increases": "increase", "increasing": "increase", "increased": "increase",
        "improve": "improve", "improves": "improve", "improving": "improve",
        "support": "support (verbo)", "supports": "support (verbo)", "supporting": "support (verbo)", "supported": "support (verbo)",
        "ensure": "ensure", "ensures": "ensure", "ensuring": "ensure", "ensured": "ensure",
        "strengthen": "strengthen", "strengthens": "strengthen", "strengthening": "strengthen", "strengthened": "strengthen",
        "foster": "foster", "fosters": "foster", "fostering": "foster", "fostered": "foster",
        "establish": "establish", "establishes": "establish", "establishing": "establish", "established": "establish",
        "expand": "expand", "expands": "expand", "expanding": "expand", "expanded": "expand",
        "create": "create (verbo)", "creates": "create (verbo)", "creating": "create (verbo)", "created": "create (verbo)",
        "develop": "develop (verbo)", "develops": "develop (verbo)", "developing": "develop (verbo)", "developed": "develop (verbo)",
        "implement": "implement (verbo)", "implements": "implement (verbo)", "implementing": "implement (verbo)", "implemented": "implement (verbo)",
        "include": "include", "includes": "include", "including": "include", "included": "include",
        "identify": "identify", "identifies": "identify", "identifying": "identify", "identified": "identify",
        "require": "require", "requires": "require", "requiring": "require", "required": "require",
        "guarantee": "guarantee", "guarantees": "guarantee", "guaranteeing": "guarantee",
        "monitor": "monitor", "monitors": "monitor", "monitoring": "monitor", "monitored": "monitor",
        "integrate": "integrate", "integrates": "integrate", "integrating": "integrate", "integrated": "integrate",
        "propose": "propose", "proposes": "propose", "proposing": "propose", "proposed": "propose",
        "protect": "protect", "protects": "protect", "protecting": "protect", "protected": "protect",
        "provide": "provide (verbo)", "provides": "provide (verbo)", "providing": "provide (verbo)", "provided": "provide (verbo)",
        "reduce": "reduce", "reduces": "reduce", "reducing": "reduce", "reduced": "reduce",
        "launch": "launch", "launches": "launch", "launching": "launch", "launched": "launch",
        "offer": "offer", "offers": "offer", "offering": "offer", "offered": "offer",
    }
    canon = {**PLURAL_MERGE, **VERB_MERGE}

    freq = Counter()
    for t in tokens:
        if t in placeholder_map:
            freq[placeholder_map[t]] += 1
            continue
        if len(t) == 1:
            continue
        if t in LIST_MARKERS:
            continue
        if t in STOPWORDS:
            continue
        if t in TEMPLATE_TERMS:
            continue
        freq[canon.get(t, t)] += 1
    return freq


In [ ]:
docs_raw = {
    "ai_plus": freq_ai_plus(),
    "new_generation": freq_new_generation(),
    "americas": freq_americas(),
    "apply_ai": freq_apply_ai(),
    "ai_continent": freq_ai_continent(),
    "pbia": freq_pbia(),
}

DOC_LABELS = {
    "ai_plus": '"AI+" Initiative (China, 2025)',
    "new_generation": "New Generation AI Development Plan (China, 2017)",
    "americas": "America's AI Action Plan (EUA, 2025)",
    "apply_ai": "Apply AI Strategy (UE, 2025)",
    "ai_continent": "AI Continent Action Plan (UE, 2025)",
    "pbia": "PBIA (Brasil, 2025)",
}

resumo = pd.DataFrame([
    {"documento": DOC_LABELS[n], "tokens_de_conteúdo": sum(f.values()), "termos_distintos": len(f)}
    for n, f in docs_raw.items()
])
resumo

## Etapa 3-c: exclusão de identificadores idiossincráticos de nacionalidade/instituição

Conforme o item 3-c da Skill B, termos que **apenas nomeiam a identidade nacional/institucional do próprio
documento** são excluídos da base de similaridade (nunca das visualizações de frequência por documento,
que permanecem intocadas). A decisão foi tomada **caso a caso**, verificando o contexto real de cada termo
em cada documento — nunca por uma lista fixa aplicada indiscriminadamente. A tabela abaixo documenta cada
termo avaliado, a decisão e a justificativa; termos mantidos (não excluídos) também são registrados, para
transparência quanto à decisão inversa.

**Casos mantidos deliberadamente (não excluídos), com justificativa:**
- `China` / `Chinese`, quando mencionados em `americas_ai_action_plan.json` e `ai_continent_action_plan.json` —
  são referências a um **terceiro país/concorrente estratégico**, não autoidentificação do documento de
  origem; carregam conteúdo temático real (enquadramento de rivalidade/competição geopolítica), mesma lógica
  do exemplo "ally/enemy" da Skill B (item 3, alínea c).
- `America` / `EU` / `European`, quando mencionados no PBIA — idem: são referências comparativas a outros
  atores internacionais (ex.: "Latin America", "the European Union... AI Observatory"), não autoidentificação
  do Brasil.
- `Member States` (UE) — não é autoidentificação, mas uma categoria de governança substantiva (a forma como
  a UE organiza poder entre suas nações constituintes), comparável a "estados subnacionais" nos EUA ou à
  estrutura federativa no PBIA — eixo de comparação legítimo, não mero rótulo de identidade.
- `Trump` (EUA, 14 ocorrências) — nome próprio de pessoa, fora do escopo estrito do item 3-c (que trata de
  identificadores de **nacionalidade/instituição**, não de indivíduos); mantido por conservadorismo
  metodológico, registrado aqui como caso de fronteira explicitamente decidido, não omitido.

In [ ]:
EXCLUDE_3C = {
    "ai_plus": {"china", "chinese", "state council"},
    "new_generation": {"china", "chinese", "state council"},
    "americas": {
        "united states", "america", "american", "americans",
        "doc", "dod", "nist", "caisi", "nsf", "doe", "dol", "omb", "ostp", "ic",
    },
    "apply_ai": {"eu", "european", "europe"},
    "ai_continent": {"eu", "european", "europe"},
    "pbia": {"brazil", "brazilian", "brasil", "mcti", "cgee", "nib", "pbia"},
}

JUSTIFICATIVA_3C = {
    "china": "país estrangeiro citado como concorrente/referência (mantido nos docs onde é terceiro; excluído nos docs do próprio país)",
    "chinese": "idem, forma adjetival",
    "state council": "nome da instituição que emite o documento (国务院) — análogo a DOC/DOD/NIST nos EUA",
    "united states": "autoidentificação do país emissor do documento (EUA)",
    "america": "autoidentificação do país emissor (EUA) — distinto de referências a 'Latin America' no PBIA",
    "american": "forma adjetival da autoidentificação (EUA)",
    "americans": "forma nominal plural da autoidentificação (EUA)",
    "doc": "Department of Commerce — agência do governo emissor (EUA)",
    "dod": "Department of Defense — agência do governo emissor (EUA)",
    "nist": "National Institute of Standards and Technology — agência do governo emissor (EUA)",
    "caisi": "Center for AI Standards and Innovation — agência do governo emissor (EUA)",
    "nsf": "National Science Foundation — agência do governo emissor (EUA)",
    "doe": "Department of Energy — agência do governo emissor (EUA)",
    "dol": "Department of Labor — agência do governo emissor (EUA)",
    "omb": "Office of Management and Budget — agência do governo emissor (EUA)",
    "ostp": "Office of Science and Technology Policy — agência do governo emissor (EUA)",
    "ic": "Intelligence Community — comunidade de inteligência do governo emissor (EUA)",
    "eu": "autoidentificação do bloco emissor (UE)",
    "european": "forma adjetival da autoidentificação (UE)",
    "europe": "forma nominal da autoidentificação (UE)",
    "brazil": "autoidentificação do país emissor (Brasil)",
    "brazilian": "forma adjetival da autoidentificação (Brasil)",
    "brasil": "autoidentificação do país emissor, grafia em português residual (citação/nome de programa)",
    "mcti": "Ministério da Ciência, Tecnologia e Inovação — instituição emissora (Brasil)",
    "cgee": "Centro de Gestão e Estudos Estratégicos — instituição responsável pela redação (Brasil)",
    "nib": "Nova Indústria Brasil — nome de programa nacional específico (Brasil)",
    "pbia": "nome/sigla do próprio documento (autorreferência)",
}

linhas = []
for doc, termos in EXCLUDE_3C.items():
    freq = docs_raw[doc]
    for termo in sorted(termos):
        ocorr = next((v for k, v in freq.items() if k.lower() == termo), 0)
        linhas.append({
            "documento": DOC_LABELS[doc], "termo_excluído": termo, "ocorrências": ocorr,
            "justificativa": JUSTIFICATIVA_3C.get(termo, ""),
        })

tabela_exclusao_3c = pd.DataFrame(linhas).sort_values(["documento", "ocorrências"], ascending=[True, False])
tabela_exclusao_3c

## Achado crítico: harmonização de rótulos entre documentos

Antes de calcular a similaridade, uma verificação sistemática de consistência (item 3 da Skill B — "usar
rótulos diferentes por documento introduziria viés na comparação") comparou, para cada conceito
compartilhado entre dois ou mais documentos, o rótulo efetivamente usado em cada pipeline individual.
Essa verificação estende a auditoria de consistência já registrada na Seção 9 (ou equivalente) de cada
`registro_vocabulario_*.md`, que havia corrigido bigramas **ausentes** em alguns documentos, mas não havia
verificado se bigramas **já presentes em múltiplos documentos** usavam exatamente o mesmo rótulo (mesma
grafia/caixa) entre si.

Essa verificação revelou uma inconsistência de alto impacto, não capturada pela auditoria anterior:

| Conceito | Rótulo em 5 dos 6 documentos | Rótulo no PBIA | Ocorrências no PBIA | Efeito sem correção |
|---|---|---|---|---|
| Inteligência Artificial | `"AI"` | `"Artificial Intelligence (AI)"` | 625 (5,8% do vocabulário do PBIA) | O termo mais frequente de **todos** os seis documentos ficava em duas dimensões vetoriais diferentes — o PBIA tinha similaridade zero com os outros cinco documentos **neste único termo**, que sozinho representa a maior fatia do vocabulário de cada documento (3,3%–6,5%) |

Duas inconsistências adicionais, de menor magnitude, foram encontradas e corrigidas pelo mesmo motivo
(diferença de caixa/grafia para o mesmo referente, sem qualquer dúvida de julgamento envolvida):

| Conceito | Documentos com rótulo `"Public/Private Sector"` (maiúsculo) | Documentos com rótulo `"public/private sector"` (minúsculo) |
|---|---|---|
| Setor Público | PBIA (19 ocorrências) | Apply AI Strategy (12), AI Continent Action Plan (18) |
| Setor Privado | EUA (9), PBIA (8) | AI Continent Action Plan (3) |

**Correção aplicada (exclusiva desta análise conjunta — não altera os notebooks/registros individuais,
cujos próprios gráficos de Top 50 continuam corretos e inalterados para exibição de cada documento
isoladamente):** os rótulos acima foram unificados sob uma única forma canônica antes da construção dos
vetores de similaridade. Esta correção é, isoladamente, a mudança de maior impacto sobre os resultados
deste notebook — sem ela, a similaridade do PBIA com os demais documentos seria artificialmente
subestimada (ver comparação "antes/depois" na célula seguinte).

In [ ]:
HARMONIZE = {
    "pbia": {"Artificial Intelligence (AI)": "AI"},
    "apply_ai": {"public sector": "Public Sector"},
    "ai_continent": {"public sector": "Public Sector", "private sector": "Private Sector"},
}

def harmonize(nome_doc, freq):
    mapa = HARMONIZE.get(nome_doc, {})
    if not mapa:
        return freq
    saida = Counter()
    for termo, n in freq.items():
        saida[mapa.get(termo, termo)] += n
    return saida

docs_harmonizados = {nome: harmonize(nome, freq) for nome, freq in docs_raw.items()}

print("Verificação pós-harmonização — ocorrências do rótulo canônico em cada documento:")
for nome, freq in docs_harmonizados.items():
    print(f"  {nome:16}: AI={freq.get('AI', 0):4d}  Public Sector={freq.get('Public Sector', 0):3d}  Private Sector={freq.get('Private Sector', 0):3d}")

## Etapas 4 e 6: vocabulário completo, vetor de frequência relativa e similaridade de cosseno

A base de similaridade é o vocabulário completo pós-etapa 3 (item 5-a da Skill B): cada documento, após
exclusão dos identificadores idiossincráticos (etapa 3-c) e harmonização de rótulos, é representado como
um vetor de **frequência relativa** — a proporção de cada termo em relação ao total de tokens de conteúdo
do próprio documento (etapa 4, normalização pela extensão do documento) — no espaço vetorial da **união**
dos vocabulários dos seis documentos. Nenhum corte por rank é aplicado em nenhuma etapa deste cálculo.

In [ ]:
docs_finais = {}
for nome, freq in docs_harmonizados.items():
    excl = EXCLUDE_3C[nome]
    docs_finais[nome] = Counter({t: n for t, n in freq.items() if t.lower() not in excl})

vocabulario_uniao = sorted(set().union(*[set(c.keys()) for c in docs_finais.values()]))
print(f"Tamanho do vocabulário-união (base de similaridade): {len(vocabulario_uniao)} termos distintos")

def vetor_freq_relativa(counter):
    total = sum(counter.values())
    return np.array([counter.get(t, 0) / total for t in vocabulario_uniao])

vetores = {nome: vetor_freq_relativa(c) for nome, c in docs_finais.items()}

def similaridade_cosseno(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

nomes = list(vetores.keys())
matriz = pd.DataFrame(
    [[similaridade_cosseno(vetores[i], vetores[j]) for j in nomes] for i in nomes],
    index=[DOC_LABELS[n] for n in nomes], columns=[DOC_LABELS[n] for n in nomes],
)
matriz.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(matriz.values, cmap="RdYlBu_r", vmin=0, vmax=1)

ax.set_xticks(range(len(nomes))); ax.set_yticks(range(len(nomes)))
ax.set_xticklabels(matriz.columns, rotation=40, ha="right", fontsize=8.5)
ax.set_yticklabels(matriz.index, fontsize=8.5)

for i in range(len(nomes)):
    for j in range(len(nomes)):
        val = matriz.values[i, j]
        cor_texto = "white" if val > 0.75 or val < 0.25 else "#222222"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9, color=cor_texto)

ax.set_title(
    "Matriz de similaridade de cosseno — vocabulário completo pós-etapa 3\n"
    "(sem corte de Top 50; identificadores nacionais/institucionais excluídos — Skill 02_Análise_Vocab_B)",
    fontsize=11, fontweight="bold", pad=14,
)
cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label("similaridade de cosseno", fontsize=9)
fig.text(0.02, -0.02,
         "Base: vetores de frequência relativa sobre o vocabulário completo de cada documento (item 5-a/6 da Skill 02_Análise_Vocab_B).",
         fontsize=7.5, color="#666666")
fig.tight_layout()
plt.show()

## Item 6.1: sequência estruturada da comparação

A leitura dos resultados segue a ordem exigida pela Skill B — do mais específico (coerência interna de
cada bloco) para o mais amplo (posicionamento do PBIA em relação ao conjunto) — e não a ordem inversa.

### 1. Coerência interna por bloco/país

In [ ]:
coerencia_china = similaridade_cosseno(vetores["ai_plus"], vetores["new_generation"])
coerencia_ue = similaridade_cosseno(vetores["apply_ai"], vetores["ai_continent"])

print(f"China — \"AI+\" Initiative (2025) vs. New Generation AI Development Plan (2017): {coerencia_china:.4f}")
print(f"União Europeia — Apply AI Strategy vs. AI Continent Action Plan (ambos 2025): {coerencia_ue:.4f}")

Os dois documentos chineses guardam coerência interna alta (0,76), mas **inferior** à dos dois documentos
europeus (0,86) — os mais internamente alinhados de todo o conjunto. Uma leitura plausível, e que o gráfico
por si só sustenta apenas parcialmente (é preciso ler o conteúdo para confirmar), é o hiato temporal: os dois
documentos europeus foram publicados no mesmo ano (2025), enquanto os dois chineses estão separados por oito
anos (2017 e 2025) — tempo suficiente para mudança de vocabulário tecnológico (ex.: a ascensão de "modelos de
linguagem" e "IA generativa" após 2017). O gráfico não permite, isoladamente, distinguir se a menor coerência
chinesa decorre do hiato temporal ou de uma divergência substantiva de enquadramento entre os dois documentos;
qualquer uma das duas leituras exigiria inspeção adicional do conteúdo, não apenas da matriz de similaridade.

### 2. Comparação entre blocos/países

In [ ]:
china_agregado = docs_finais["ai_plus"] + docs_finais["new_generation"]
ue_agregado = docs_finais["apply_ai"] + docs_finais["ai_continent"]

vetores_bloco = {
    "China (agregado)": vetor_freq_relativa(china_agregado),
    "UE (agregado)": vetor_freq_relativa(ue_agregado),
    "EUA": vetores["americas"],
}
nomes_bloco = list(vetores_bloco.keys())
matriz_bloco = pd.DataFrame(
    [[similaridade_cosseno(vetores_bloco[i], vetores_bloco[j]) for j in nomes_bloco] for i in nomes_bloco],
    index=nomes_bloco, columns=nomes_bloco,
)
matriz_bloco.round(3)

Entre os três blocos/países com múltiplas abordagens documentadas, **UE e EUA** apresentam a maior
similaridade (0,79), **China e EUA** ficam em posição intermediária (0,72) e **China e UE** apresentam a
menor similaridade cruzada (0,68) — a China é, portanto, o bloco mais distinto vocabularmente dos outros
dois, tanto em relação à UE quanto (em menor grau) em relação aos EUA.

### 3. Posicionamento do PBIA

In [ ]:
pbia_vs_cada = pd.Series({
    DOC_LABELS[n]: similaridade_cosseno(vetores["pbia"], vetores[n])
    for n in ["ai_plus", "new_generation", "americas", "apply_ai", "ai_continent"]
}).sort_values(ascending=False)

centroide_outros5 = np.mean([vetores[n] for n in ["ai_plus", "new_generation", "americas", "apply_ai", "ai_continent"]], axis=0)
pbia_vs_centroide = similaridade_cosseno(vetores["pbia"], centroide_outros5)

print("PBIA vs. cada documento (do mais ao menos similar):")
print(pbia_vs_cada.round(4).to_string())
print(f"\nPBIA vs. centroide simples dos outros cinco documentos: {pbia_vs_centroide:.4f}")
print(f"(referência: similaridade média entre pares dos outros cinco documentos, excluindo a diagonal: "
      f"{np.mean([similaridade_cosseno(vetores[a], vetores[b]) for a in ['ai_plus','new_generation','americas','apply_ai','ai_continent'] for b in ['ai_plus','new_generation','americas','apply_ai','ai_continent'] if a < b]):.4f})")

O PBIA aproxima-se mais dos dois documentos europeus e do documento estadunidense (0,71–0,74) do que dos
dois documentos chineses (0,53–0,62) — sendo o "AI+" Initiative chinês (2025) o documento do qual o PBIA
mais se distancia de todo o conjunto. A similaridade do PBIA com o centroide dos outros cinco documentos
(0,77) é próxima da similaridade média entre pares dentro desses mesmos cinco documentos — ou seja, **após
a correção do rótulo "AI"/"Artificial Intelligence (AI)"**, o PBIA deixa de aparecer como um documento
vocabularmente isolado (o que a versão não corrigida da análise sugeriria de forma enganosa) e passa a se
posicionar dentro da mesma ordem de grandeza de similaridade observada entre os demais documentos —
próximo, mas não indistinto, do conjunto EUA/UE.

## Riqueza lexical por documento (evidência empírica da inadequação do Top 50 para similaridade)

A tabela abaixo evidencia, com dados deste próprio corpus, por que um corte fixo de Top 50 teria cobertura
desigual entre os seis documentos (item 5-a da Skill B, razão 3): quanto maior a razão *types/tokens*
(vocabulário mais diverso), menor a fração da massa total de tokens que um corte fixo de 50 termos consegue
capturar.

In [ ]:
linhas_riqueza = []
for nome, c in docs_finais.items():
    total = sum(c.values())
    distintos = len(c)
    massa_top50 = sum(n for _, n in c.most_common(50))
    linhas_riqueza.append({
        "documento": DOC_LABELS[nome],
        "tokens_de_conteúdo": total,
        "termos_distintos": distintos,
        "razão_types_tokens": round(distintos / total, 3),
        "%_da_massa_coberta_pelo_Top_50": round(massa_top50 / total * 100, 1),
    })

tabela_riqueza = pd.DataFrame(linhas_riqueza).sort_values("razão_types_tokens", ascending=False)
tabela_riqueza

A cobertura do Top 50 varia de 27,2% (AI Continent Action Plan) a 37,6% (\"AI+\" Initiative) da massa total
de cada documento — uma diferença de mais de 10 pontos percentuais entre os documentos, que se somaria aos
outros dois problemas já discutidos (zeros artificiais assimétricos e viés a favor do vocabulário genérico)
caso o Top 50 fosse usado como base do cálculo de similaridade em vez do vocabulário completo.

## Visualizações de frequência por documento (base 5-b — Top 15, apenas para contexto de leitura)

As visualizações abaixo usam a **outra** base vocabular (item 5-b da Skill B — corte de Top 50, aqui exibido
em Top 15 por espaço) e servem apenas de referência rápida para interpretar **quais termos concretos**
sustentam as similaridades/diferenças acima. Diferentemente da matriz de similaridade, este painel usa o
vocabulário **bruto** de cada documento (`docs_raw`) — **sem** a exclusão de identificadores nacionais/
institucionais (etapa 3-c) nem a harmonização de rótulos, ambas exclusivas da base de similaridade (item 5-a).
Isso é proposital: identificadores como "Brazil"/"China"/"EU" são vocabulário legítimo e informativo na
leitura de frequência de **um único documento** (Skill 02_Análise_Vocab_A) — só distorcem a **comparação**
entre documentos (item 3-c), não a leitura interna de cada um. Os gráficos de Top 25/Top 50 completos e
definitivos de cada documento permanecem nos respectivos notebooks individuais de cada país/bloco, que não
são alterados por esta análise conjunta; o painel abaixo é apenas um recorte de apoio à leitura, no mesmo
notebook onde a similaridade é calculada.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
cores_doc = {
    "ai_plus": "#a11d21", "new_generation": "#8c1d1d", "americas": "#1f4e79",
    "apply_ai": "#003399", "ai_continent": "#1f4e79", "pbia": "#2a78d6",
}

for ax, nome in zip(axes.flat, docs_raw.keys()):
    top15 = docs_raw[nome].most_common(15)[::-1]
    termos = [t for t, _ in top15]
    valores = [v for _, v in top15]
    ax.barh(termos, valores, color=cores_doc[nome], height=0.7)
    ax.set_title(DOC_LABELS[nome], fontsize=10, fontweight="bold")
    ax.tick_params(axis="y", labelsize=7.5)
    ax.tick_params(axis="x", labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Top 15 termos por documento — vocabulário bruto pós-etapa 3 (item 5-b), SEM exclusão 3-c nem harmonização\n"
             "(painel de apoio à leitura — não é a base usada na matriz de similaridade, que aplica ambas as camadas)",
             fontsize=11.5, fontweight="bold", y=1.02)
fig.tight_layout()
plt.show()

## Análise final obrigatória (item 10 da Skill B)

### Metodologia aplicada
- **Idiomas:** todos os seis documentos estão em inglês em `texto_completo` — comparação lexical direta
  válida, sem etapa de equivalência entre idiomas.
- **Documento central:** nenhum documento foi tratado automaticamente como referência; a sequência
  estruturada (item 6.1) posiciona o PBIA por último, por ser o eixo declarado da pesquisa (ver
  "Documento central da comparação" na Skill B), não por suposição não verificada.
- **Remoção/normalização/bigramas/hífen/desambiguação:** integralmente herdados dos seis notebooks
  individuais, já auditados sob a Skill 02_Análise_Vocab_A (ver `registro_vocabulario_*.md` de cada
  documento) — nenhuma dessas decisões foi alterada aqui.
- **Exclusão de identificadores idiossincráticos (etapa 3-c):** aplicada caso a caso a 27 termos no total
  (ver tabela na Seção "Etapa 3-c"), com casos de fronteira (China/EU/America citados como terceiros;
  Member States; Trump) deliberadamente mantidos e justificados.
- **Harmonização de rótulos entre documentos:** achado crítico do rótulo "AI" vs. "Artificial Intelligence
  (AI)" no PBIA (o termo mais frequente de todo o corpus) corrigido, além de duas inconsistências menores de
  caixa em "Public/Private Sector".
- **Base vocabular da similaridade:** vocabulário completo pós-etapa 3 (item 5-a), sem corte de rank —
  jamais o Top 50 (item 5-b, usado apenas nas visualizações de frequência por documento, Seção anterior).
- **Normalização relativa:** frequência de cada termo dividida pelo total de tokens de conteúdo do próprio
  documento (item 4).
- **Métrica de similaridade:** similaridade de cosseno entre vetores de frequência relativa, no espaço da
  união dos vocabulários dos seis documentos (item 6).
- **Sequência estruturada seguida (item 6.1):** coerência interna China → coerência interna UE → comparação
  entre blocos (China agregado / UE agregado / EUA) → posicionamento do PBIA.

### O que os gráficos mostram e como interpretá-los
A matriz de similaridade de cosseno mede a proximidade do **perfil vocabular relativo** de cada par de
documentos — valores próximos de 1 indicam vocabulários proporcionalmente muito parecidos (mesmos termos
ocupando fatias semelhantes do conteúdo de cada documento); valores próximos de 0 indicam vocabulários
proporcionalmente muito distintos. A tabela de riqueza lexical mostra por que o Top 50 seria uma base
inadequada para essa mesma medição. Os pequenos múltiplos de Top 15 servem apenas para inspecionar quais
termos concretos aparecem nos documentos mais/menos similares entre si — não para medir similaridade.

### Onde os documentos se aproximam e se diferenciam (estritamente dentro do que a matriz sustenta)
- **UE e EUA** são o par de blocos mais próximo entre os três com múltiplos documentos/abordagens (0,79).
- **China** é o bloco mais distinto tanto da UE (0,68) quanto, em menor grau, dos EUA (0,72).
- Internamente, a **UE é mais coesa** (0,86 entre seus dois documentos) do que a **China** (0,76 entre os
  seus) — possivelmente relacionado ao hiato de 8 anos entre os dois documentos chineses, mas a matriz por
  si só não decide entre essa hipótese e uma divergência substantiva de enquadramento entre os dois planos
  chineses; isso exigiria leitura adicional de conteúdo.
- O **PBIA** aproxima-se mais da UE e dos EUA (0,71–0,74) do que da China (0,53–0,62), e sua similaridade
  ao centroide dos outros cinco documentos (0,77) é da mesma ordem de grandeza da similaridade média entre
  esses cinco documentos entre si — ou seja, o PBIA não é um documento vocabularmente isolado do conjunto,
  ao contrário do que uma análise sem a correção do rótulo "AI" sugeriria.

### Ajustes possíveis para melhorar a visualização
- Adicionar uma matriz de distância/dendrograma (agrupamento hierárquico) para visualizar a estrutura de
  blocos de forma mais imediata do que a leitura numérica da matriz — não implementado aqui por ausência da
  biblioteca `scipy` neste ambiente (ver limitação, abaixo).
- Ponderar os vetores por uma medida tipo TF-IDF (em vez de frequência relativa pura), para dar peso maior
  aos termos que discriminam entre documentos e peso menor ao vocabulário genérico do gênero textual — uma
  extensão explicitamente compatível com a Skill B (item 6, segundo parágrafo: "isso não substitui outras
  formas de análise... que o usuário venha a solicitar de modo complementar").

### Limitações e lacunas remanescentes (a serem preenchidas)
- **Exclusão de identificadores (3-c) concentrada em candidatos de alta/média frequência.** A varredura
  cobriu os termos já sinalizados nos registros individuais mais uma busca direcionada por nomes de país/
  bloco, siglas de agências e nome do próprio plano — não uma auditoria termo a termo de toda a cauda longa
  de cada vocabulário (750 a 2.280 termos distintos por documento). É possível que outros identificadores
  de baixíssima frequência (ex.: nomes de cidades, de outros programas nacionais específicos) permaneçam não
  excluídos; dado o baixo peso relativo de termos de frequência 1-2, o efeito esperado sobre a matriz de
  similaridade é pequeno, mas não nulo.
- **Harmonização de rótulos também não exaustiva.** Foram corrigidas as inconsistências de alto e médio
  impacto identificadas (AI/Artificial Intelligence; Public/Private Sector). Cinco inconsistências residuais
  de impacto muito baixo (≤2 ocorrências cada, <0,03% do vocabulário de cada documento) foram identificadas
  e deliberadamente **não corrigidas** nesta rodada, por não alterarem nenhum resultado de forma perceptível:
  "Research & Development" não protegido em `ai_continent_action_plan.json` (1 ocorrência bruta); "Machine
  Learning" não protegido em `americas_ai_action_plan.json` (1 ocorrência); "Data Center" não protegido em
  `new_generation_ai_development_plan.json` (1 ocorrência); "Public Service" e "Value Chain" não protegidos
  em `apply_ai_strategy.json` (1 ocorrência cada). Formalizar essas cinco correções nos notebooks individuais
  de origem — replicando o mesmo padrão de bigrama já usado nos demais documentos — é uma lacuna de baixo
  risco e baixo esforço que pode ser preenchida em uma iteração futura, caso se deseje precisão máxima.
- **Sem biblioteca de agrupamento hierárquico (`scipy`) neste ambiente.** A leitura de "quais documentos
  formam clusters" foi feita manualmente a partir da matriz numérica (item 6.1), não de um dendrograma
  formal. Instalar `scipy` no ambiente do projeto resolveria essa lacuna sem exigir nenhuma mudança
  metodológica.
- **Tradução como limitação estrutural inerente (herdada da Skill02/Skill B, item 1).** Quatro dos seis
  documentos (os dois chineses e, em menor medida, o PBIA) chegam a este corpus como texto **traduzido**
  para o inglês por terceiros (CSET, New America/DigiChina, MCTI/CGEE) — toda tradução introduz imprecisão
  adicional em relação ao texto original, uma limitação que nenhuma etapa metodológica desta skill pode
  eliminar, apenas registrar.
- **Similaridade por frequência relativa pura, não por TF-IDF.** Conforme já mencionado nos ajustes
  possíveis, o método aqui é o padrão explicitamente definido pela Skill B (item 6) — uma ponderação
  tipo TF-IDF é uma complementação legítima, não uma correção de erro, e não foi implementada por não ter
  sido solicitada.

---

# Extensão: TF-IDF, dendrograma e mapa de similaridade (scipy)

**As células acima não foram modificadas.** Tudo a partir daqui é adicionado por cima do resultado já
obtido, para corrigir três lacunas explicitamente registradas na Seção "Limitações e lacunas remanescentes":
(1) ausência de `scipy` / dendrograma formal; (2) similaridade calculada apenas por frequência relativa
pura, sem ponderação por relevância discriminante (TF-IDF); (3) ausência de um mapa visual 2D de
posicionamento entre os documentos.

Esta extensão **não substitui** a matriz de similaridade já calculada acima (item 6 da Skill B, frequência
relativa pura) — ela constrói, em paralelo, uma segunda base de comparação ponderada por TF-IDF, mais
adequada para identificar o que **distingue** os documentos entre si (em vez de apenas o que eles têm em
comum), e usa essa segunda base para o dendrograma e o mapa 2D pedidos. As duas bases são comparadas
explicitamente, para que fique claro o que muda e por quê.

## TF-IDF: metodologia

A frequência relativa pura (usada na matriz de similaridade original, acima) trata todo termo do mesmo jeito
independentemente de quantos dos seis documentos o compartilham — por isso um termo presente nos **seis**
documentos (como "AI") pesa tanto quanto um termo presente em apenas **um** deles, mesmo que o primeiro não
ajude em nada a diferenciar os documentos entre si (todos falam de IA o tempo todo; esse é o próprio
critério de seleção do corpus) e o segundo seja exatamente o tipo de termo que **discrimina** um documento
dos demais.

TF-IDF corrige isso multiplicando a frequência relativa de cada termo (TF) por um fator (IDF) que **cresce
conforme o termo aparece em menos documentos do corpus**:

$$\text{tfidf}(t, d) = \text{tf}(t, d) \times \text{idf}(t), \qquad
\text{idf}(t) = \ln\!\left(\frac{1 + N}{1 + \text{df}(t)}\right) + 1$$

onde $N = 6$ (número de documentos do corpus) e $\text{df}(t)$ é quantos desses seis documentos contêm o
termo $t$ ao menos uma vez. Esta é a fórmula **suavizada** padrão (equivalente ao `TfidfVectorizer` do
scikit-learn com `smooth_idf=True`, aqui reimplementada manualmente por não haver `scikit-learn` neste
ambiente) — a suavização (+1 no numerador e denominador, +1 externo) evita que um termo presente em todos os
seis documentos receba peso exatamente zero (o que aconteceria com a fórmula não suavizada
$\ln(N/\text{df})$), preservando ainda alguma informação sobre a proporção relativa desse termo entre
documentos, em vez de descartá-la por completo.

A base vocabular usada é a mesma já estabelecida acima — vocabulário completo pós-etapa 3, após exclusão
3-c e harmonização de rótulos (item 5-a da Skill B) —, sem nenhum corte de rank.

In [ ]:
from scipy.linalg import eigh
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram, cophenet

N_DOCS = len(vocabulario_uniao) and len(docs_finais)

# document frequency de cada termo entre os 6 documentos (quantos documentos contêm o termo)
df_termo = np.array([
    sum(1 for c in docs_finais.values() if c.get(t, 0) > 0) for t in vocabulario_uniao
])
idf = np.log((1 + N_DOCS) / (1 + df_termo)) + 1

vetores_tfidf = {nome: vetores[nome] * idf for nome in nomes}

matriz_tfidf = pd.DataFrame(
    [[similaridade_cosseno(vetores_tfidf[i], vetores_tfidf[j]) for j in nomes] for i in nomes],
    index=[DOC_LABELS[n] for n in nomes], columns=[DOC_LABELS[n] for n in nomes],
)

idx_ai = vocabulario_uniao.index("AI")
print(f"Checagem do IDF — 'AI' aparece em {int(df_termo[idx_ai])}/6 documentos -> idf = {idf[idx_ai]:.3f} "
      f"(termo presente em todos os documentos, peso reduzido mas não anulado pela suavização)")
idx_rara = int(np.argmax(df_termo == 1))
print(f"Exemplo de termo em apenas 1 documento ({vocabulario_uniao[idx_rara]!r}): idf = {idf[idx_rara]:.3f} "
      f"(peso máximo — mais discriminante)")
print()
matriz_tfidf.round(3)

### Comparação: similaridade por frequência relativa pura vs. por TF-IDF

A tabela abaixo mostra as duas bases lado a lado. Como esperado, **todas** as similaridades caem sob
TF-IDF — isso é o efeito pretendido, não um erro: o vocabulário genérico compartilhado por todo o corpus
(que inflava a similaridade de frequência pura) perde peso, restando principalmente o vocabulário que de
fato diferencia os documentos entre si.

In [ ]:
comparacao = []
for i, a in enumerate(nomes):
    for b in nomes[i + 1:]:
        comparacao.append({
            "par": f"{DOC_LABELS[a]} × {DOC_LABELS[b]}",
            "freq. relativa pura": round(similaridade_cosseno(vetores[a], vetores[b]), 4),
            "TF-IDF": round(similaridade_cosseno(vetores_tfidf[a], vetores_tfidf[b]), 4),
        })
tabela_comparacao = pd.DataFrame(comparacao)
tabela_comparacao["Δ (TF-IDF − freq. pura)"] = (tabela_comparacao["TF-IDF"] - tabela_comparacao["freq. relativa pura"]).round(4)
tabela_comparacao.sort_values("TF-IDF", ascending=False)

A ordem geral dos pares muda pouco (os pares mais/menos similares continuam, em linhas gerais, os mesmos),
mas há uma inversão relevante para a leitura do PBIA: sob frequência pura, o PBIA era mais próximo do
**America's AI Action Plan** (0,7217) do que do **AI Continent Action Plan** (0,7117); sob TF-IDF, essa
ordem se **inverte** — o PBIA passa a ser mais próximo do **AI Continent Action Plan** (0,4602) do que do
documento estadunidense (0,4547), e o **Apply AI Strategy** segue sendo, nas duas bases, o documento mais
próximo do PBIA. Ou seja: uma vez removido o peso do vocabulário genérico compartilhado por todo o corpus,
o PBIA aparece consistentemente mais alinhado aos **dois** documentos europeus do que ao documento
estadunidense — um resultado mais nítido do que o sugerido pela frequência pura.

## Dendrograma (agrupamento hierárquico, `scipy.cluster.hierarchy`)

O dendrograma usa a mesma base TF-IDF, convertida em distância euclidiana **exata** — não uma aproximação —
a partir dos vetores TF-IDF normalizados para norma unitária: para vetores unitários,
$\lVert a - b \rVert^2 = 2(1 - \cos(a, b))$ é uma identidade algébrica exata, não uma estimativa. Essa é
a mesma matriz de distância usada no mapa 2D da próxima seção, para que as duas visualizações sejam
consistentes entre si. O método de ligação é **average linkage** (média das distâncias entre todos os pares
de elementos de dois grupos) — escolhido, entre `average`/`complete`/`ward`, por ser o que produziu a maior
**correlação cofenética** (medida de quão bem o dendrograma preserva as distâncias originais: 1,0 = perfeito)
para este conjunto específico de seis documentos, testado explicitamente abaixo.

In [ ]:
# vetores TF-IDF normalizados para norma unitária -> distância euclidiana exatamente derivada do cosseno
vetores_tfidf_unit = {n: vetores_tfidf[n] / np.linalg.norm(vetores_tfidf[n]) for n in nomes}

dist_matriz = np.array([
    [np.linalg.norm(vetores_tfidf_unit[a] - vetores_tfidf_unit[b]) for b in nomes] for a in nomes
])
dist_condensada = squareform(dist_matriz, checks=False)

print("Correlação cofenética por método de ligação (quanto mais perto de 1,0, mais fiel o dendrograma):")
melhores = {}
for metodo in ["average", "complete", "ward"]:
    Z_teste = linkage(dist_condensada, method=metodo)
    corr, _ = cophenet(Z_teste, dist_condensada)
    melhores[metodo] = corr
    print(f"  {metodo:10}: {corr:.4f}")

metodo_escolhido = max(melhores, key=melhores.get)
Z = linkage(dist_condensada, method=metodo_escolhido)
print(f"\nMétodo escolhido: {metodo_escolhido} (correlação cofenética = {melhores[metodo_escolhido]:.4f})")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
dendrogram(
    Z, labels=[DOC_LABELS[n] for n in nomes], ax=ax,
    color_threshold=0.85, leaf_font_size=9.5,
)
ax.set_ylabel("distância euclidiana (derivada do cosseno TF-IDF)", fontsize=9.5)
ax.set_title(
    f"Dendrograma — agrupamento hierárquico por similaridade TF-IDF (ligação: {metodo_escolhido})\n"
    f"correlação cofenética = {melhores[metodo_escolhido]:.3f}",
    fontsize=11.5, fontweight="bold", pad=12,
)
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=15, ha="right")
fig.tight_layout()
plt.show()

**Leitura do dendrograma.** A altura em que dois ramos se unem indica a distância entre eles — uniões mais
baixas (mais próximas da base) indicam maior similaridade. Os dois documentos europeus se unem primeiro (a
menor altura de todo o dendrograma), confirmando numericamente o que a matriz já mostrava: são o par mais
coeso do corpus. Os dois documentos chineses formam o segundo par a se unir. O documento dos EUA se junta ao
par europeu antes de qualquer um dos dois se juntar ao par chinês — reproduzindo, em forma de árvore, a
mesma hierarquia já lida na matriz (UE e EUA mais próximos entre si do que qualquer um dos dois está da
China). O PBIA só se junta ao restante do conjunto por último, no nível mais alto (maior distância) da
árvore — consistente com o TF-IDF ter aumentado a distância relativa do PBIA a todos os demais documentos
(Seção anterior), mas ainda assim mais próximo do ramo europeu/estadunidense do que do ramo chinês, se se
observar a ordem em que os ramos aparecem da esquerda para a direita.

## Mapa de similaridade 2D (escalonamento multidimensional clássico)

### Metodologia

Um "mapa" que respeita fielmente as similaridades de todos os $\binom{6}{2}=15$ pares simultaneamente
exigiria, em geral, até 5 dimensões (6 documentos podem ocupar, no máximo, um espaço de 5 dimensões
independentes). Para reduzir a **2 dimensões** — o que um mapa efetivamente exige — usa-se **escalonamento
multidimensional clássico** (*classical/metric MDS*, Torgerson-Gower), implementado aqui diretamente com
`scipy.linalg.eigh` (não há `MDS` pronto no `scipy`; essa função existe no `scikit-learn`, indisponível neste
ambiente, mas o método clássico é inteiramente redutível a uma decomposição espectral, que o `scipy` faz):

1. Parte-se da mesma matriz de distância euclidiana exata (derivada do cosseno TF-IDF) usada no dendrograma.
2. Aplica-se **duplo centramento** à matriz de distâncias ao quadrado, obtendo uma matriz de produtos internos $B$.
3. Decompõe-se $B$ em autovalores/autovetores (`scipy.linalg.eigh`, por $B$ ser simétrica).
4. As coordenadas 2D de cada documento são os dois primeiros autovetores, escalados pela raiz de seus
   respectivos autovalores — a solução que **minimiza a distorção** entre a distância real (6D) e a
   distância representada no mapa (2D), para qualquer mapa 2D possível.

### Por que é preciso declarar o quanto o mapa distorce a realidade

Reduzir de 5 dimensões reais para 2 necessariamente descarta informação. Antes de interpretar qualquer
posição no mapa, é preciso quantificar **quanto** se perde — do contrário, o mapa pode sugerir proximidades
ou distâncias que não correspondem à similaridade real entre os documentos, exatamente o tipo de imprecisão
que compromete a leitura dos fatos. Duas medidas-padrão são calculadas antes de desenhar o mapa: a
**variância explicada** pelas duas primeiras dimensões (quanto da estrutura total de distâncias as 2
dimensões capturam) e o **stress-1 de Kruskal** (o erro de reconstrução das distâncias reais a partir do
mapa 2D; por convenção, stress < 0,05 é excelente, < 0,10 bom, < 0,20 razoável, e ≥ 0,20 indica um ajuste
pobre, a ser lido com cautela).

In [ ]:
D = dist_matriz  # reaproveita a matriz de distância exata já calculada para o dendrograma
D2 = D ** 2
n = len(nomes)
J = np.eye(n) - np.ones((n, n)) / n
B = -0.5 * J @ D2 @ J

autovalores, autovetores = eigh(B)
ordem = np.argsort(autovalores)[::-1]
autovalores, autovetores = autovalores[ordem], autovetores[:, ordem]

print("Scree — variância explicada e stress-1 de Kruskal por número de dimensões da MDS:")
soma_autovalores_positivos = autovalores[autovalores > 0].sum()
for k in range(1, 6):
    Xk = autovetores[:, :k] * np.sqrt(np.clip(autovalores[:k], 0, None))
    Dk = np.sqrt(((Xk[:, None, :] - Xk[None, :, :]) ** 2).sum(-1))
    stress_k = np.sqrt(np.sum((D - Dk) ** 2) / np.sum(D ** 2))
    var_exp_k = autovalores[:k].sum() / soma_autovalores_positivos
    marcador = "  <- usado no mapa abaixo" if k == 2 else ""
    print(f"  k={k}: variância explicada={var_exp_k*100:5.1f}%   stress-1={stress_k:.4f}{marcador}")

# coordenadas finais em 2D
X2D = autovetores[:, :2] * np.sqrt(np.clip(autovalores[:2], 0, None))
D_2d = np.sqrt(((X2D[:, None, :] - X2D[None, :, :]) ** 2).sum(-1))
stress_2d = np.sqrt(np.sum((D - D_2d) ** 2) / np.sum(D ** 2))
var_exp_2d = autovalores[:2].sum() / soma_autovalores_positivos

coords_mds = pd.DataFrame(X2D, index=[DOC_LABELS[n] for n in nomes], columns=["dim_1", "dim_2"])
print(f"\nMapa final (2D): variância explicada = {var_exp_2d*100:.1f}% | stress-1 = {stress_2d:.4f} "
      f"({'AVISO: ajuste pobre, ver limitações abaixo' if stress_2d >= 0.2 else 'ajuste aceitável'})")
coords_mds

In [ ]:
cores_bloco = {
    "ai_plus": "#a11d21", "new_generation": "#8c1d1d",
    "americas": "#1f4e79",
    "apply_ai": "#003399", "ai_continent": "#0059b3",
    "pbia": "#2a9d3f",
}
marcadores_bloco = {
    "ai_plus": "o", "new_generation": "o", "americas": "s", "apply_ai": "^", "ai_continent": "^", "pbia": "D",
}

fig, ax = plt.subplots(figsize=(8.5, 7.5))
for nome in nomes:
    x, y = X2D[nomes.index(nome)]
    ax.scatter(x, y, s=260, color=cores_bloco[nome], marker=marcadores_bloco[nome],
               edgecolor="white", linewidth=1.3, zorder=3)
    ax.annotate(DOC_LABELS[nome], (x, y), textcoords="offset points", xytext=(9, 7), fontsize=9.5)

ax.axhline(0, color="#cccccc", linewidth=0.8, zorder=1)
ax.axvline(0, color="#cccccc", linewidth=0.8, zorder=1)
ax.set_xlabel("dimensão 1 (MDS)", fontsize=10)
ax.set_ylabel("dimensão 2 (MDS)", fontsize=10)
ax.set_title(
    "Mapa de similaridade — escalonamento multidimensional clássico sobre distância TF-IDF\n"
    f"variância explicada pelas 2 dimensões: {var_exp_2d*100:.1f}%  |  stress-1 de Kruskal: {stress_2d:.3f} "
    "(ajuste pobre — ver interpretação abaixo)",
    fontsize=11, fontweight="bold", pad=14,
)
ax.set_aspect("equal", adjustable="datalim")
fig.text(0.02, -0.02,
         "Distâncias ENTRE clusters (China / UE+EUA / PBIA) são relativamente confiáveis neste mapa; "
         "distâncias DENTRO de cada par (China↔China; UE↔UE) são artificialmente comprimidas — ver tabela de checagem, abaixo.",
         fontsize=8, color="#8a3a00")
fig.tight_layout()
plt.show()

### Por que cada documento está onde está — e onde o mapa engana

**Leitura do posicionamento.** No eixo horizontal (dimensão 1), o mapa separa claramente um polo chinês
(`"AI+"` e New Generation, à esquerda) de um polo europeu/estadunidense (à direita) — essa é a dimensão
dominante de divergência vocabular do corpus, coerente com a China ser o bloco mais distinto tanto da UE
quanto dos EUA já identificado na matriz numérica. O PBIA se destaca na dimensão vertical (dimensão 2),
isolado dos demais nesse eixo — o que é exatamente o esperado, dado que o TF-IDF tornou o PBIA o documento
mais distante de **todos** os outros cinco (Seção "Posicionamento do PBIA", acima), não apenas de um bloco
específico: o mapa captura essa distância generalizada deslocando o PBIA para uma dimensão própria, em vez
de aproximá-lo de qualquer polo específico.

**Onde o mapa distorce a realidade (leitura obrigatória antes de tirar conclusões visuais).** O stress-1 de
0,376 (ajuste "pobre" pelo critério de Kruskal) não é uniforme: ele se concentra especificamente nas
distâncias **dentro** de cada par de mesmo bloco. A tabela abaixo compara a distância real (6D) com a
distância reconstruída no mapa (2D) para os 15 pares — repare que os dois pares intra-bloco (China↔China e
UE↔UE) são os mais comprimidos em termos proporcionais, enquanto os pares que envolvem o PBIA (o documento
mais afastado de todos) são os mais bem preservados:

In [ ]:
linhas_diag = []
for i in range(len(nomes)):
    for j in range(i + 1, len(nomes)):
        real, mapa = D[i, j], D_2d[i, j]
        linhas_diag.append({
            "par": f"{DOC_LABELS[nomes[i]]} × {DOC_LABELS[nomes[j]]}",
            "distância real (6D)": round(real, 3),
            "distância no mapa (2D)": round(mapa, 3),
            "compressão": f"{(1 - mapa / real) * 100:.0f}%",
        })
tabela_diag = pd.DataFrame(linhas_diag)
tabela_diag.sort_values("distância real (6D)")

Os pares "AI+ × New Generation" (China) e "Apply AI × AI Continent" (UE) são comprimidos em **80% e 92%**,
respectivamente — no mapa, os dois documentos europeus quase se sobrepõem visualmente, o que **não** significa
que sejam quase idênticos (a similaridade de cosseno real entre eles é 0,76, não 1,0). Isso acontece porque
a MDS prioriza, matematicamente, preservar as distâncias **maiores** da matriz (as que mais pesam na soma de
erro quadrático que a MDS minimiza) — como as distâncias envolvendo o PBIA são as maiores de todo o corpus,
o mapa "gasta" a maior parte da sua capacidade de representação em posicionar bem o PBIA, sacrificando a
precisão das distâncias menores entre pares do mesmo bloco. **Conclusão prática de leitura:** este mapa é
confiável para julgar agrupamentos amplos (quais documentos formam "famílias" distantes umas das outras) —
e, por esse critério, o dendrograma da seção anterior (correlação cofenética de 0,91) é a visualização
**mais fiel** das duas para esse mesmo propósito — mas não deve ser usado para comparar visualmente o quão
parecidos são dois documentos do mesmo bloco entre si; para isso, a matriz numérica de similaridade
(Seções anteriores) continua sendo a fonte correta.

## Mapa centrado no PBIA — contraste com cada documento

### Metodologia

O mapa 2D acima é um compromisso simultâneo entre os 15 pares do corpus — e, como acabou de ser demonstrado,
esse compromisso comprime especificamente as distâncias que **não** envolvem o PBIA. Para responder à
pergunta específica "como o PBIA se posiciona em contraste com cada documento", a visualização mais fiel
**não** é reaproveitar as coordenadas 2D acima (que introduziriam a mesma distorção já diagnosticada), mas
plotar diretamente as **cinco distâncias reais** entre o PBIA e cada um dos outros documentos — cada uma
delas é um número exato (não uma aproximação de projeção), lido diretamente da matriz de similaridade TF-IDF.
O diagrama radial abaixo usa o PBIA como origem (centro = distância zero); cada um dos outros cinco
documentos é posicionado em um raio proporcional à sua **dissimilaridade** em relação ao PBIA
($1 - \cos_{\text{tfidf}}$: 0 = idêntico, valores maiores = mais distante) — quanto mais perto do centro,
mais parecido com o PBIA. O ângulo de cada documento não carrega significado métrico (é apenas um
agrupamento visual por bloco/país, para facilitar a leitura), diferentemente do raio, que é exato.

In [ ]:
outros = [n for n in nomes if n != "pbia"]
dissimilaridade_pbia = {n: 1 - similaridade_cosseno(vetores_tfidf["pbia"], vetores_tfidf[n]) for n in outros}

# agrupamento angular por bloco, para leitura visual (não carrega significado métrico)
angulos_deg = {
    "apply_ai": 20, "ai_continent": 60,          # UE, lado direito
    "americas": 130,                              # EUA, topo
    "new_generation": 200, "ai_plus": 240,        # China, esquerda
}
angulos_rad = {n: np.deg2rad(a) for n, a in angulos_deg.items()}

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": "polar"})
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)

raio_max = max(dissimilaridade_pbia.values()) * 1.25
for n in outros:
    r = dissimilaridade_pbia[n]
    theta = angulos_rad[n]
    ax.plot([theta, theta], [0, r], color=cores_bloco[n], linewidth=2, zorder=2)
    ax.scatter([theta], [r], s=280, color=cores_bloco[n], marker=marcadores_bloco[n],
               edgecolor="white", linewidth=1.3, zorder=3)
    ax.annotate(f"{DOC_LABELS[n]}\n(dissimilaridade = {r:.3f})", (theta, r),
                textcoords="offset points", xytext=(0, 14), ha="center", fontsize=8.8)

ax.scatter([0], [0], s=380, color=cores_bloco["pbia"], marker=marcadores_bloco["pbia"],
           edgecolor="white", linewidth=1.5, zorder=4)
ax.annotate("PBIA\n(centro, dissimilaridade = 0)", (0, 0), textcoords="offset points",
            xytext=(0, -32), ha="center", fontsize=9.5, fontweight="bold")

ax.set_ylim(0, raio_max)
ax.set_yticklabels([])
ax.set_xticklabels([])
ax.grid(True, alpha=0.35)
ax.set_title(
    "PBIA como centro de análise — dissimilaridade TF-IDF exata a cada documento\n"
    "(raio = 1 − similaridade de cosseno; ângulo apenas agrupa visualmente por bloco, sem significado métrico)",
    fontsize=11, fontweight="bold", pad=22,
)
fig.tight_layout()
plt.show()

### O que faz o PBIA se posicionar onde está

Do mais próximo ao mais distante do PBIA (TF-IDF): **Apply AI Strategy** (0,499) < **AI Continent Action
Plan** (0,540) < **America's AI Action Plan** (0,545) < **New Generation AI Development Plan** (0,602) <
**"AI+" Initiative** (0,668). Inspecionando os termos que mais contribuem para essas distâncias (produto
elemento a elemento dos vetores TF-IDF, o componente que efetivamente entra no cálculo do cosseno):

- **PBIA × Apply AI Strategy (mais próximo):** além de "AI" (esperado em qualquer par), os termos que mais
  contribuem para a proximidade são vocabulário de política pública genérico mas ainda relativamente pouco
  compartilhado no corpus como um todo — "use", "data", "develop", "potential", "public", "strategic" —
  sugerindo que a proximidade do PBIA com o Apply AI Strategy vem de um registro de política pública
  comparativamente similar (estratégico, orientado a aplicação/adoção), não de um único tema dominante.
- **PBIA × "AI+" Initiative (mais distante):** os termos que mais contribuem para a (baixa) similaridade
  residual são "development", "promote", "new", "strengthen", "industry" — vocabulário de mobilização
  industrial típico do documento chinês, que tem peso pequeno mas não nulo no PBIA. Em contrapartida, os
  termos mais fortes no PBIA e **ausentes** no "AI+" Initiative são "action(s)", "impact(s)", "challenge(s)",
  "technology/technologies", "innovation(s)", "solution(s)", "system(s)" — vocabulário estrutural do formato
  do PBIA (um plano organizado em "Ações", "Impactos" e "Desafios" numerados, com anexos extensos), que **não
  tem equivalente estrutural direto** no texto do "AI+" Initiative, um documento muito mais curto e em
  formato de comunicado normativo.

Esse último ponto é uma ressalva metodológica importante: parte da distintividade lexical do PBIA em relação
aos demais documentos não é puramente temática, mas reflete o **gênero textual/formato** do próprio
documento (plano de ação extenso, com centenas de itens numerados em anexo) — uma característica já
registrada em `pbia_vocab_registro.md` (Seção 1.3) para a análise individual do documento, e que se propaga,
de forma esperada, para a análise comparada.

## Limitações desta extensão e melhorias possíveis

**Limitações da extensão TF-IDF:**
- O corpus tem apenas **6 documentos** — para uma estatística de "quantos documentos contêm este termo"
  (document frequency), 6 é uma amostra muito pequena: o IDF só assume 6 valores possíveis (df = 1 a 6), uma
  resolução grosseira. Com mais documentos no corpus (ex.: outros planos nacionais de IA não incluídos
  aqui), o IDF ganharia granularidade e passaria a discriminar melhor graus intermediários de raridade.
- A suavização escolhida (`smooth_idf`, padrão scikit-learn) é uma escolha razoável e citável, mas não a
  única válida — outras variantes (IDF probabilístico, BM25) ponderariam os termos de forma um pouco
  diferente; nenhuma foi testada como alternativa aqui.

**Limitações do mapa 2D (MDS):**
- **Stress-1 = 0,376 — ajuste pobre pelo critério convencional de Kruskal.** Isso já foi destacado no corpo
  do notebook, mas vale repetir aqui como limitação central desta extensão: com apenas 6 pontos, um mapa 2D
  comprime desproporcionalmente as distâncias intra-bloco (demonstrado na tabela de checagem). O mapa deve
  ser lido como um **esboço aproximado** de agrupamento amplo, nunca como fonte de leitura fina de distâncias.
- O dendrograma (correlação cofenética 0,91) é objetivamente mais fiel à estrutura real de distâncias do que
  o mapa 2D (variância explicada de apenas 58,3%) — em qualquer aplicação futura desta metodologia, o
  dendrograma deveria ser tratado como a visualização primária de agrupamento, e o mapa 2D como um
  complemento ilustrativo, não o contrário.
- Uma melhoria direta e de baixo custo seria acrescentar um **mapa 3D** (ou um segundo mapa 2D nas dimensões
  3 e 4) — a variância explicada sobe para 76,9% com 3 dimensões e 90,3% com 4 — o que reduziria
  substancialmente a distorção, ao custo de um gráfico menos imediato de ler que um mapa 2D plano.

**Limitações do mapa radial centrado no PBIA:**
- Os ângulos de cada documento foram escolhidos manualmente para fins de legibilidade (agrupar visualmente
  por bloco) — eles **não** carregam nenhum significado métrico e não devem ser lidos como uma segunda
  dimensão de comparação; apenas o raio (distância exata ao PBIA) é informativo. Isso está declarado no
  título do gráfico, mas é um ponto de atenção permanente para quem for reproduzir ou adaptar este gráfico.
- O diagrama radial informa apenas as distâncias **entre o PBIA e cada um dos outros cinco documentos** — ele
  não representa as distâncias entre os outros cinco documentos entre si (ex.: entre os dois documentos
  chineses); para isso, a matriz de similaridade completa (Seções anteriores) continua sendo necessária.

**Melhorias possíveis para o notebook como um todo, além do já implementado nesta extensão:**
- Adicionar `scikit-learn` ao ambiente permitiria comparar esta implementação manual de TF-IDF/MDS com a
  implementação de referência da biblioteca, como checagem cruzada de corretude.
- Expandir o corpus além dos 6 documentos atuais (outros planos nacionais de IA, ex.: Reino Unido, Índia —
  já há uma pasta `Índia` no projeto, ainda sem JSON extraído) tornaria o IDF mais granular e o mapa/
  dendrograma mais informativos, com mais pontos para ancorar a estrutura de distância.
- Formalizar, nos notebooks individuais de cada documento, as cinco correções de bigrama de impacto
  negligenciável já identificadas e registradas como lacuna na análise anterior (Research & Development em
  `ai_continent_action_plan.json`; Machine Learning em `americas_ai_action_plan.json`; Data Center em
  `new_generation_ai_development_plan.json`; Public Service e Value Chain em `apply_ai_strategy.json`).